In [1]:
# Optimized Biomarker Selection Algorithm 
# Run the module/function Optimized_Biomarker_Selection to recursively eliminate low-impact biomarkers 
# and select highly significant biomarkers for cancer detection.
# Applied on local calibration sample of the dataset

import pandas as pd
import numpy as np
import gudhi
import os
from sklearn.linear_model import LinearRegression

def Optimized_Biomarker_Selection(
    distance_threshold_offset,
    tumor_types,  # Dictionary with cancer types and number of patients to consider
    max_patients,
    normal_data_size,
    top_n_files_to_eliminate,
    cancer_types,  # List of cancer types to consider
    Select_Biomarkers,
    clinical_data_path,
    sheet_name,
    feature_vector_path,
    output_folder_path,
    FPR  # False positive rate
):
    # Calculate required_normal_positives based on normal_data_size
    required_normal_positives = int(normal_data_size * FPR)
    print(f"Calculated required normal positives: {required_normal_positives} (from {normal_data_size} normal patients)")

    # Part 1: Perform TDA on the clinical data and save the results to a CSV file
    def part_1():
        file_path = clinical_data_path
        df_1 = pd.read_excel(file_path, sheet_name=sheet_name)
        df_2 = pd.read_excel(file_path, sheet_name=sheet_name)
        
        # Check if Select_Biomarkers exist in the DataFrame
        missing_columns = [col for col in Select_Biomarkers if col not in df_1.columns]
        if missing_columns:
            raise ValueError(f"The following columns are missing in the dataset: {missing_columns}")
        
        df_1 = df_1[Select_Biomarkers]
        df_2 = df_2[Select_Biomarkers]
        
        df_1 = df_1.iloc[:59]
        data_list = []
        
        # Get the clinical data for cohort information
        df_clinical = pd.read_excel(file_path, sheet_name=sheet_name)
        additional_columns = ['Patient ID #', 'Cohort', 'AJCC Stage']
        df_additional_info = df_clinical[additional_columns].iloc[60:]
        
        # Create a dictionary to track how many patients we've processed for each tumor type
        patients_processed = {tumor_type.lower(): 0 for tumor_type in tumor_types}
        
        # Process patients starting from row 60 until we've reached all required counts
        patient_counter = 0
        processed_indices = []  # Track which indices we're processing
        for index in range(60, max_patients + 1):
            # Check if we've processed enough patients for all tumor types
            if all(count >= tumor_types[tumor_type] for tumor_type, count in patients_processed.items()):
                break
                
            # Get the cohort for this patient
            cohort = str(df_additional_info.iloc[index-60]['Cohort']).strip().lower()
            
            # Skip if this patient's cohort isn't in our tumor_types or we've already processed enough
            if cohort not in patients_processed or patients_processed[cohort] >= tumor_types[cohort]:
                continue
                
            # Process this patient
            row_to_add = df_2.iloc[[index]]
            df_new = pd.concat([df_1] + [row_to_add] * 1, ignore_index=True)
            
            numeric_cols = df_new.select_dtypes(include=[np.number]).columns
            df_new[numeric_cols] = np.log(df_new[numeric_cols])
            
            column_names = df_new.columns.tolist()
            df_subset = df_new.values
            
            corr_matrix = np.corrcoef(df_subset, rowvar=False)
            corr_matrix_df = pd.DataFrame(corr_matrix, columns=column_names, index=column_names)
            distance_matrix = 1 - np.abs(corr_matrix_df)
            distance_matrix_np = distance_matrix.values
            
            rips_complex = gudhi.RipsComplex(distance_matrix=distance_matrix_np)
            simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
            persistence = simplex_tree.persistence()
            
            finite_persistence = [(dim, (birth, death)) for dim, (birth, death) in persistence if death != float('inf')]
            one_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 1]
            
            if one_dim_holes:
                max_life_range_1D = max(death - birth for birth, death in one_dim_holes)
                avg_birth_1D = np.mean([birth for birth, _ in one_dim_holes])
                avg_death_1D = np.mean([death for _, death in one_dim_holes])
                num_one_dim_holes = len(one_dim_holes)
            else:
                max_life_range_1D = avg_birth_1D = avg_death_1D = num_one_dim_holes = 0
            
            zero_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 0]
            min_life_range_0D = min(death - birth for birth, death in zero_dim_holes)
            max_life_range_0D = max(death - birth for birth, death in zero_dim_holes)
            avg_death_0D = np.mean([death for _, death in zero_dim_holes])
            num_zero_dim_holes = len(zero_dim_holes) + 1
            range_death_0D = max([death for _, death in zero_dim_holes]) - min([death for _, death in zero_dim_holes])
            median_death_0D = np.median([death for _, death in zero_dim_holes])
            std_death_0D = np.std([death for _, death in zero_dim_holes])
            
            data = {
                "0-Dim Hole Min Life Range": [min_life_range_0D],
                "0-Dim Hole Max Life Range": [max_life_range_0D],
                "Range_Death_0D": [range_death_0D],
                "Num_0D_Holes": [num_zero_dim_holes],
                "Median_Death_0D": [median_death_0D],
                "Std_Death_0D": [std_death_0D],
                "Avg_Death_0D": [avg_death_0D],
                "Average Birth (1D)": [avg_birth_1D],
                "Average Death (1D)": [avg_death_1D],
                "1-Dim Hole Max Life Range": [max_life_range_1D],
                "Number of 1-Dimensional Holes": [num_one_dim_holes]
            }
            
            df = pd.DataFrame(data, index=[f"P_{index}"])
            data_list.append(df)
            patients_processed[cohort] += 1
            patient_counter += 1
            processed_indices.append(index)
        
        final_df = pd.concat(data_list)
        
        # Add clinical information for only the processed patients
        df_additional_info = df_clinical[additional_columns].iloc[processed_indices]
        df_additional_info.index = [f"P_{i}" for i in processed_indices]
        
        df_combined = pd.concat([df_additional_info, final_df], axis=1)
        df_combined.to_csv(feature_vector_path, float_format="%f", index=True)
        print(f"CSV file '{feature_vector_path}' has been created with patient information for {patient_counter} patients.")
        return processed_indices  # Return the processed indices for use in part_3
    
    # Part 2: Construct a linear regression model and analyze the data
    def part_2():
        file_path = feature_vector_path
        new_data = pd.read_csv(file_path, header=0)
        
        def construct_linear_regression_model():
            # Use only the specified normal_data_size patients for training
            data = new_data[new_data['Cohort'].str.strip().str.lower() == 'normal'].iloc[:normal_data_size]
            X = data[['0-Dim Hole Min Life Range', '0-Dim Hole Max Life Range', 'Median_Death_0D', 'Std_Death_0D']]
            y = data['Avg_Death_0D']
            
            model = LinearRegression()
            model.fit(X, y)
            
            coefficients = model.coef_
            intercept = model.intercept_
            
            print(f"\nRegression Equation: Avg_Death_0D = {coefficients[0]:.4f} * 0-Dim Hole Min Life Range + "
                  f"{coefficients[1]:.4f} * 0-Dim Hole Max Life Range + "
                  f"{coefficients[2]:.4f} * Median_Death_0D + "
                  f"{coefficients[3]:.4f} * Std_Death_0D + "
                  f"{intercept:.4f}")
            
            return model, coefficients, intercept
        
        def predict_and_analyze():
            _, coefficients, intercept = construct_linear_regression_model()
            
            def predict_Avg_Death_0D(row):
                return (coefficients[0] * row['0-Dim Hole Min Life Range'] +
                        coefficients[1] * row['0-Dim Hole Max Life Range'] +
                        coefficients[2] * row['Median_Death_0D'] +
                        coefficients[3] * row['Std_Death_0D'] +
                        intercept)
            
            new_data['Predicted Avg_Death_0D'] = new_data.apply(predict_Avg_Death_0D, axis=1)
            new_data['Difference'] = new_data['Avg_Death_0D'] - new_data['Predicted Avg_Death_0D']
            new_data['Regression Difference Percentage'] = abs(new_data['Difference'] / new_data['Avg_Death_0D']) * 100
            
            # Calculate reference point from normal data
            Normal_data = new_data[new_data['Cohort'].str.strip().str.lower() == 'normal'].iloc[:normal_data_size]
            reference_point = (Normal_data['Average Birth (1D)'].mean(), Normal_data['Average Death (1D)'].mean())
            
            print(f"\nReference Point (Normal Cohort): {reference_point}")
            
            def euclidean_distance(x1, y1, x2, y2):
                return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
            
            new_data['Distance %(1)'] = new_data.apply(
                lambda row: euclidean_distance(row['Average Birth (1D)'], row['Average Death (1D)'], reference_point[0], reference_point[1]) * 100,
                axis=1
            )
            
            # Calculate averages for each tumor type (only up to specified patient counts)
            averages_list = []
            columns_to_average = [
                '0-Dim Hole Min Life Range', 'Range_Death_0D', '1-Dim Hole Max Life Range',
                'Average Birth (1D)', 'Average Death (1D)', 
                'Regression Difference Percentage', 'Distance %(1)'
            ]
            
            for tumor, row_limit in tumor_types.items():
                filtered_data = new_data[new_data['Cohort'].str.strip().str.lower() == tumor]
                selected_data = filtered_data.iloc[:row_limit]
                selected_data = selected_data[columns_to_average].dropna()
                column_averages = selected_data.mean(axis=0)
                averages_list.append({'Cohort': tumor.capitalize(), **column_averages.to_dict()})
            
            averages_data = pd.DataFrame(averages_list)
            new_data.to_csv(file_path, index=False)
            
            # Calculate Normal distance threshold
            Normal_row = averages_data[averages_data['Cohort'].str.strip().str.lower() == 'normal']
            if not Normal_row.empty:
                Normal_distance_threshold = Normal_row['Distance %(1)'].values[0] + distance_threshold_offset
                print(f"\nFinal 'Distance %(1)' Threshold (k_2): {Normal_distance_threshold:.6f}")
            else:
                print("Error: 'Normal' cohort not found in averages_data.")
                return None
            
            # Find optimal threshold that gives required_normal_positives in normal cohort
            threshold = 0
            max_threshold = 1.0
            best_threshold = None
            best_counts = None
            
            while threshold < max_threshold:
                new_data['Result(Cancer)'] = new_data.apply(
                    lambda row: 'Positive' if sum([
                        row['Regression Difference Percentage'] > threshold,
                        row['Distance %(1)'] > Normal_distance_threshold
                    ]) >= 2 else 'Negative',
                    axis=1
                )
                
                # Count positives in normal cohort (only up to normal_data_size)
                normal_data = new_data[new_data['Cohort'].str.strip().str.lower() == 'normal'].iloc[:normal_data_size]
                Normal_positive = sum(normal_data['Result(Cancer)'] == 'Positive')
                
                if Normal_positive == required_normal_positives:
                    best_threshold = threshold
                    # Count positives for all tumor types (up to their specified limits)
                    positive_counts = {}
                    total_positive_cancers = 0
                    
                    for tumor_type, patient_count in tumor_types.items():
                        if tumor_type.lower() != 'normal':
                            tumor_data = new_data[new_data['Cohort'].str.strip().str.lower() == tumor_type.lower()]
                            tumor_data = tumor_data.iloc[:patient_count]
                            positive_counts[tumor_type] = sum(tumor_data['Result(Cancer)'] == 'Positive')
                            total_positive_cancers += positive_counts[tumor_type]
                    
                    best_counts = positive_counts
                    break
                
                threshold += 0.001
            
            if best_threshold is not None:
                print(f"\nFinal Regression Difference Threshold (k_1): {best_threshold:.4f}")
                print("Positive counts per cancer type:")
                for tumor, count in best_counts.items():
                    print(f"{tumor}: {count}")
                print(f"Total Positive Cancers: {total_positive_cancers}")
                return Normal_distance_threshold
            else:
                print("\nRequired result for the given threshold is not obtained.")
                return None
        
        Normal_threshold = predict_and_analyze()
        print(f"\nNormal Threshold (k_2): {Normal_threshold}")
        return Normal_threshold
    
    # Part 3: Loop through biomarkers, leaving one out at a time
    def part_3(processed_indices):
        file_path = clinical_data_path
        df_1 = pd.read_excel(file_path, sheet_name=sheet_name)
        df_2 = pd.read_excel(file_path, sheet_name=sheet_name)
        
        additional_columns = ['Patient ID #', 'Cohort', 'AJCC Stage']
        df_clinical = pd.read_excel(file_path, sheet_name=sheet_name)
        
        # Use the processed_indices from part_1
        df_additional_info = df_clinical[additional_columns].iloc[processed_indices]
        df_additional_info.index = [f"P_{i}" for i in processed_indices]
        
        biomarker_file_map = {}
        file_biomarker_map = {}
        
        for i, biomarker_to_leave in enumerate(Select_Biomarkers):
            selected_columns = [b for b in Select_Biomarkers if b != biomarker_to_leave]
            
            df_1_subset = df_1[selected_columns].iloc[:59]
            data_list = []
            
            # Process the same patients we processed in part_1
            for index in processed_indices:
                row_to_add = df_2[selected_columns].iloc[[index]]
                df_new = pd.concat([df_1_subset] + [row_to_add] * 1, ignore_index=True)
                
                numeric_cols = df_new.select_dtypes(include=[np.number]).columns
                df_new[numeric_cols] = np.log(df_new[numeric_cols])
                
                column_names = df_new.columns.tolist()
                df_subset = df_new.values
                
                corr_matrix = np.corrcoef(df_subset, rowvar=False)
                corr_matrix_df = pd.DataFrame(corr_matrix, columns=column_names, index=column_names)
                distance_matrix = 1 - np.abs(corr_matrix_df)
                
                rips_complex = gudhi.RipsComplex(distance_matrix=distance_matrix.values)
                simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
                persistence = simplex_tree.persistence()
                
                finite_persistence = [(dim, (birth, death)) for dim, (birth, death) in persistence if death != float('inf')]
                one_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 1]
                
                if one_dim_holes:
                    max_life_range_1D = max(death - birth for birth, death in one_dim_holes)
                    avg_birth_1D = np.mean([birth for birth, _ in one_dim_holes])
                    avg_death_1D = np.mean([death for _, death in one_dim_holes])
                    num_one_dim_holes = len(one_dim_holes)
                else:
                    max_life_range_1D = avg_birth_1D = avg_death_1D = num_one_dim_holes = 0
                
                zero_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 0]
                min_life_range_0D = min(death - birth for birth, death in zero_dim_holes)
                max_life_range_0D = max(death - birth for birth, death in zero_dim_holes)
                avg_death_0D = np.mean([death for _, death in zero_dim_holes])
                num_zero_dim_holes = len(zero_dim_holes) + 1
                range_death_0D = max([death for _, death in zero_dim_holes]) - min([death for _, death in zero_dim_holes])
                median_death_0D = np.median([death for _, death in zero_dim_holes])
                std_death_0D = np.std([death for _, death in zero_dim_holes])
                
                data = {
                    "0-Dim Hole Min Life Range": [min_life_range_0D],
                    "0-Dim Hole Max Life Range": [max_life_range_0D],
                    "Range_Death_0D": [range_death_0D],
                    "Num_0D_Holes": [num_zero_dim_holes],
                    "Median_Death_0D": [median_death_0D],
                    "Std_Death_0D": [std_death_0D],
                    "Avg_Death_0D": [avg_death_0D],
                    "Average Birth (1D)": [avg_birth_1D],
                    "Average Death (1D)": [avg_death_1D],
                    "1-Dim Hole Max Life Range": [max_life_range_1D],
                    "Number of 1-Dimensional Holes": [num_one_dim_holes]
                }
                
                df = pd.DataFrame(data, index=[f"P_{index}"])
                data_list.append(df)
            
            final_df = pd.concat(data_list)
            final_df = pd.concat([df_additional_info, final_df], axis=1)
            
            csv_file_path = os.path.join(output_folder_path, f"FeatureVectorCancerDet{i + 1}.csv")
            final_df.to_csv(csv_file_path, float_format="%f", index=True)
            
            biomarker_file_map[f"FeatureVectorCancerDet{i + 1}.csv"] = biomarker_to_leave
            file_biomarker_map[csv_file_path] = biomarker_to_leave
            
            print(f"CSV file 'FeatureVectorCancerDet{i + 1}.csv' created by leaving biomarker: {biomarker_to_leave}")
        
        print("\nMapping of biomarkers to their respective CSV files:")
        print(biomarker_file_map)
        return biomarker_file_map, file_biomarker_map
    
    # Part 4: Apply regression model to modified datasets
    def part_4(biomarker_file_map, file_biomarker_map):
        folder_path = output_folder_path
        
        def construct_linear_regression_model(data):
            # Use only the specified normal_data_size patients for training
            training_data = data[data['Cohort'].str.strip().str.lower() == 'normal'].iloc[:normal_data_size]
            X = training_data[['0-Dim Hole Min Life Range', '0-Dim Hole Max Life Range', 'Median_Death_0D', 'Std_Death_0D']]
            y = training_data['Avg_Death_0D']
            
            model = LinearRegression()
            model.fit(X, y)
            
            return model, model.coef_, model.intercept_
        
        def predict_and_analyze(file_path):
            new_data = pd.read_csv(file_path, header=0)
            _, coefficients, intercept = construct_linear_regression_model(new_data)
            
            def predict_Avg_Death_0D(row):
                return (coefficients[0] * row['0-Dim Hole Min Life Range'] +
                        coefficients[1] * row['0-Dim Hole Max Life Range'] +
                        coefficients[2] * row['Median_Death_0D'] +
                        coefficients[3] * row['Std_Death_0D'] +
                        intercept)
            
            new_data['Predicted Avg_Death_0D'] = new_data.apply(predict_Avg_Death_0D, axis=1)
            new_data['Difference'] = new_data['Avg_Death_0D'] - new_data['Predicted Avg_Death_0D']
            new_data['Regression Difference Percentage'] = abs(new_data['Difference'] / new_data['Avg_Death_0D']) * 100
            
            # Calculate reference point from normal data
            control_data = new_data[new_data['Cohort'].str.strip().str.lower() == 'normal'].iloc[:normal_data_size]
            reference_point = (control_data['Average Birth (1D)'].mean(), control_data['Average Death (1D)'].mean())
            
            def euclidean_distance(x1, y1, x2, y2):
                return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
            
            new_data['Distance %(1)'] = new_data.apply(
                lambda row: euclidean_distance(row['Average Birth (1D)'], row['Average Death (1D)'], reference_point[0], reference_point[1]) * 100,
                axis=1
            )
            
            # Find threshold that gives required_normal_positives in normal cohort
            threshold = 0
            max_threshold = 1.0
            best_result = -1
            
            while threshold < max_threshold:
                new_data['Result(Cancer)'] = new_data.apply(
                    lambda row: 'Positive' if sum([
                        row['Regression Difference Percentage'] > threshold,
                        row['Distance %(1)'] > Normal_threshold,
                    ]) >= 2 else 'Negative',
                    axis=1
                )
                
                # Count positives in normal cohort (only up to normal_data_size)
                normal_data = new_data[new_data['Cohort'].str.strip().str.lower() == 'normal'].iloc[:normal_data_size]
                Normal_positive = sum(normal_data['Result(Cancer)'] == 'Positive')
                
                if Normal_positive == required_normal_positives:
                    # Count positives for all tumor types (up to their specified limits)
                    total_positive_cancers = 0
                    
                    for tumor_type, patient_count in tumor_types.items():
                        if tumor_type.lower() != 'normal':
                            tumor_data = new_data[new_data['Cohort'].str.strip().str.lower() == tumor_type.lower()]
                            tumor_data = tumor_data.iloc[:patient_count]
                            total_positive_cancers += sum(tumor_data['Result(Cancer)'] == 'Positive')
                    
                    best_result = total_positive_cancers
                    break
                
                threshold += 0.001
            
            return best_result

        positive_cancer_counts = []
        
        for i in range(1, len(Select_Biomarkers) + 1):
            file_name = f'FeatureVectorCancerDet{i}.csv'
            file_path = os.path.join(folder_path, file_name)
            
            if os.path.exists(file_path):
                result = predict_and_analyze(file_path)
                print(f"Total Positive Cancers for {file_name} is: {result}")
                positive_cancer_counts.append((file_name, result))
            else:
                print(f"File {file_name} not found.")
        
        # Sort and eliminate least significant biomarkers
        sorted_positive_cancer_counts = sorted(positive_cancer_counts, key=lambda x: x[1], reverse=True)
        top_files = sorted_positive_cancer_counts[:top_n_files_to_eliminate]
        remaining_files = [file for file in positive_cancer_counts if file not in top_files]
        
        remaining_file_names = [file_name for file_name, _ in remaining_files]
        print(f"\nRemaining CSV Files after eliminating top {top_n_files_to_eliminate} Total Positive Cancers:")
        print(remaining_file_names)
        
        remaining_biomarkers = [biomarker_file_map[file_name] for file_name in remaining_file_names if file_name in biomarker_file_map]
        print(f"The Significant Biomarkers: {remaining_biomarkers}")
        
        eliminated_biomarkers = [biomarker_file_map[file_name] for file_name, _ in top_files if file_name in biomarker_file_map]
        print(f"Eliminated least significant {top_n_files_to_eliminate} biomarkers: {eliminated_biomarkers}")
    
    # Execute the parts in sequence
    processed_indices = part_1()  # Get the indices of processed patients
    Normal_threshold = part_2()
    if Normal_threshold is not None:  # Only proceed if part_2 was successful
        biomarker_file_map, file_biomarker_map = part_3(processed_indices)  # Pass the processed indices
        part_4(biomarker_file_map, file_biomarker_map)

# Example usage (Provide the required user-defined values)
Optimized_Biomarker_Selection(
    distance_threshold_offset=1,
    tumor_types={
        'normal': 50,
        'bladder': 19,
        'ovarian': 18,
        'pancreatic': 19,
        },
    max_patients=322,
    normal_data_size=50,
    top_n_files_to_eliminate=0,
    cancer_types=['bladder', 'ovarian', 'pancreatic'],
    Select_Biomarkers=['sc-kit/SCFR', 'CA-125', 'CA 19-9', 'HE4', 'Prolactin', 'SCF', 'Total PSA', 'IGFBP3'],
    clinical_data_path=r'D:\Cancer Detection folder\Biomarker Selection\clinical cancer validation data.xlsx',
    sheet_name='Normal and Cancer',
    feature_vector_path="D:/Cancer Detection folder/Biomarker Selection/FeatureVectorCancerDet.csv",
    output_folder_path="D:/Cancer Detection folder/Biomarker Selection",
    FPR=0.05
)

Calculated required normal positives: 2 (from 50 normal patients)
CSV file 'D:/Cancer Detection folder/Biomarker Selection/FeatureVectorCancerDet.csv' has been created with patient information for 106 patients.

Regression Equation: Avg_Death_0D = 0.3337 * 0-Dim Hole Min Life Range + 0.0361 * 0-Dim Hole Max Life Range + 0.1215 * Median_Death_0D + 0.7558 * Std_Death_0D + 0.2770

Reference Point (Normal Cohort): (0.7238577, 0.7763446599999999)

Final 'Distance %(1)' Threshold (k_2): 1.830454

Final Regression Difference Threshold (k_1): 0.2860
Positive counts per cancer type:
bladder: 8
ovarian: 10
pancreatic: 15
Total Positive Cancers: 33

Normal Threshold (k_2): 1.8304535218740685
CSV file 'FeatureVectorCancerDet1.csv' created by leaving biomarker: sc-kit/SCFR
CSV file 'FeatureVectorCancerDet2.csv' created by leaving biomarker: CA-125
CSV file 'FeatureVectorCancerDet3.csv' created by leaving biomarker: CA 19-9
CSV file 'FeatureVectorCancerDet4.csv' created by leaving biomarker: HE4
CSV

In [2]:
# Get Thresholds Algorithm
# Run the function/module get_thresholds_1 to determine the final cancer detection 
# thresholds k_1 and k_2 for the selected biomarker set of the dataset by using a local calibration sample.

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# This function finalize k_1 and k_2 parameter values for further use
def get_thresholds_1(
    feature_vector_path,  # Path to the feature vector CSV file
    normal_data_size,     # Number of healthy patients in the dataset
    distance_threshold_offset,  # Offset for the distance threshold (k_2)
    cancer_types,         # List of cancer types to calculate total positive cancers
    tumor_types,          # Dictionary of tumor types and their row limits
    required_normal_positives  # User-defined required number of normal positives to maintain specificity above 95%
):
    """
    Calculate and return the thresholds:
    - Normal_distance_threshold (k_2)
    - Regression Difference Threshold (k_1)
    - Total Positive Cancers
    """
    # Load the feature vector data
    new_data = pd.read_csv(feature_vector_path, header=0)
    
    def construct_linear_regression_model():
        # Prepare the data for regression
        data = new_data.iloc[:normal_data_size]
        X = data[['0-Dim Hole Min Life Range', '0-Dim Hole Max Life Range', 'Median_Death_0D', 'Std_Death_0D']]
        y = data['Avg_Death_0D']
        
        # Fit the regression model
        model = LinearRegression()
        model.fit(X, y)
        
        # Get coefficients and intercept
        coefficients = model.coef_
        intercept = model.intercept_
        
        print(f"\nRegression Equation: Avg_Death_0D = {coefficients[0]:.4f} * 0-Dim Hole Min Life Range + "
              f"{coefficients[1]:.4f} * 0-Dim Hole Max Life Range + "
              f"{coefficients[2]:.4f} * Median_Death_0D + "
              f"{coefficients[3]:.4f} * Std_Death_0D + "
              f"{intercept:.4f}")
        
        return model, coefficients, intercept
    
    def predict_and_analyze():
        # Construct the regression model
        _, coefficients, intercept = construct_linear_regression_model()
        
        # Predict Avg_Death_0D for all rows
        def predict_Avg_Death_0D(row):
            return (coefficients[0] * row['0-Dim Hole Min Life Range'] +
                    coefficients[1] * row['0-Dim Hole Max Life Range'] +
                    coefficients[2] * row['Median_Death_0D'] +
                    coefficients[3] * row['Std_Death_0D'] +
                    intercept)
        
        new_data['Predicted Avg_Death_0D'] = new_data.apply(predict_Avg_Death_0D, axis=1)
        new_data['Difference'] = new_data['Avg_Death_0D'] - new_data['Predicted Avg_Death_0D']
        new_data['Regression Difference Percentage'] = abs(new_data['Difference'] / new_data['Avg_Death_0D']) * 100
        
        # Calculate the reference point for the normal cohort
        Normal_data = new_data.iloc[:normal_data_size]
        reference_point = (Normal_data['Average Birth (1D)'].mean(), Normal_data['Average Death (1D)'].mean())
        print(f"\nReference Point (Normal Cohort): {reference_point}")
        
        # Calculate Euclidean distance for each row
        def euclidean_distance(x1, y1, x2, y2):
            return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
        
        new_data['Distance %(1)'] = new_data.apply(
            lambda row: euclidean_distance(row['Average Birth (1D)'], row['Average Death (1D)'], reference_point[0], reference_point[1]) * 100,
            axis=1
        )
        
        # Calculate the Normal_distance_threshold (k_2)
        columns_to_average = [
            '0-Dim Hole Min Life Range',
            'Range_Death_0D',
            '1-Dim Hole Max Life Range',
            'Average Birth (1D)',
            'Average Death (1D)',
            'Regression Difference Percentage',
            'Distance %(1)'
        ]
        
        averages_list = []
        
        for tumor, row_limit in tumor_types.items():
            filtered_data = new_data[new_data['Cohort'].str.strip().str.lower() == tumor]
            selected_data = filtered_data.iloc[:row_limit]
            selected_data = selected_data[columns_to_average].dropna()
            column_averages = selected_data.mean(axis=0)
            averages_list.append({'Cohort': tumor.capitalize(), **column_averages.to_dict()})
        
        averages_data = pd.DataFrame(averages_list)
        # print("\nFinal Averages Data:")
        # print(averages_data)
        
        Normal_row = averages_data[averages_data['Cohort'].str.strip().str.lower() == 'normal']
        
        if not Normal_row.empty:
            Normal_distance_threshold = Normal_row['Distance %(1)'].values[0] + distance_threshold_offset
            print(f"\nFinal 'Distance %(1)' Threshold (k_2): {Normal_distance_threshold:.6f}")
        else:
            print("Error: 'Normal' cohort not found in averages_data.")
            return None, None, None, None
        
        # Calculate the Regression Difference Threshold (k_1)
        threshold = 0
        max_threshold = 1.0
        
        while threshold < max_threshold:
            new_data['Result(Cancer)'] = new_data.apply(
                lambda row: 'Positive' if sum([
                    row['Regression Difference Percentage'] > threshold,
                    row['Distance %(1)'] > Normal_distance_threshold
                ]) >= 2 else 'Negative',
                axis=1
            )
            
            positive_counts = new_data[new_data['Result(Cancer)'] == 'Positive'].groupby('Cohort').size()
            Normal_positive = positive_counts.get('Normal', 0)
            
            if Normal_positive == required_normal_positives:
                print(f"\nFinal Regression Difference Threshold (k_1): {threshold:.4f}")
                total_positive_cancers = sum(positive_counts.get(t, 0) for t in cancer_types)
                print(f"Total Positive Cancers: {total_positive_cancers}")
                # Return the Normal_distance_threshold, regression_threshold, total_positive_cancers, and 1-Dim Hole Max Life Range
                return Normal_distance_threshold, threshold, total_positive_cancers, Normal_row['1-Dim Hole Max Life Range'].values[0]
            
            threshold += 0.001
        
        print("\nRequired result for the given threshold is not obtained.")
        return Normal_distance_threshold, None, None, None
    
    # Call the predict_and_analyze function to get the thresholds
    Normal_distance_threshold, regression_threshold, total_positive_cancers, normal_1_dim_hole_max_life_range = predict_and_analyze()
    
    return Normal_distance_threshold, regression_threshold, total_positive_cancers, normal_1_dim_hole_max_life_range

# Example usage (Provide the required user-defined values)
feature_vector_path = "D:/Cancer Detection folder/Biomarker Selection/FeatureVectorCancerDet.csv" # This csv file is obtained from the previous code for the final selected biomarker set
normal_data_size = 50
cancer_types = ['Bladder', 'Ovarian', 'Pancreatic']
tumor_types = {'normal': 50}

# Initialize variables to store the best result
best_total_positive_cancers = 0
best_Normal_distance_threshold = 0
best_regression_threshold = 0
best_normal_1_dim_hole_max_life_range = 0

# User-defined required number of normal positives
required_normal_positives = 2  # Provide a number to maintain specificity above 95%

# Loop through distance_threshold_offset from -1 to 2 with an increment of 0.1
for distance_threshold_offset in np.arange(0.8, 2, 0.1): # User can adjust the range as per the requirement
    print(f"\nCalculating thresholds for distance_threshold_offset = {distance_threshold_offset:.1f}")
    Normal_distance_threshold, regression_threshold, total_positive_cancers, normal_1_dim_hole_max_life_range = get_thresholds_1(
        feature_vector_path, normal_data_size, distance_threshold_offset, cancer_types, tumor_types, required_normal_positives
    )
    
    if total_positive_cancers is not None and total_positive_cancers > best_total_positive_cancers:
        best_total_positive_cancers = total_positive_cancers
        best_Normal_distance_threshold = Normal_distance_threshold
        best_regression_threshold = regression_threshold
        best_normal_1_dim_hole_max_life_range = normal_1_dim_hole_max_life_range

print(f"\nBest Result:")
print(f"Highest Total Positive Cancers: {best_total_positive_cancers}")
print(f"Normal Distance Threshold (k_2): {best_Normal_distance_threshold}")
print(f"Regression Difference Threshold (k_1): {best_regression_threshold}")
print(f"1-Dim Hole Max Life Range (Normal Cohort): {best_normal_1_dim_hole_max_life_range}")


Calculating thresholds for distance_threshold_offset = 0.8

Regression Equation: Avg_Death_0D = 0.3337 * 0-Dim Hole Min Life Range + 0.0361 * 0-Dim Hole Max Life Range + 0.1215 * Median_Death_0D + 0.7558 * Std_Death_0D + 0.2770

Reference Point (Normal Cohort): (0.7238577, 0.7763446599999999)

Final 'Distance %(1)' Threshold (k_2): 1.630454

Final Regression Difference Threshold (k_1): 0.2860
Total Positive Cancers: 35

Calculating thresholds for distance_threshold_offset = 0.9

Regression Equation: Avg_Death_0D = 0.3337 * 0-Dim Hole Min Life Range + 0.0361 * 0-Dim Hole Max Life Range + 0.1215 * Median_Death_0D + 0.7558 * Std_Death_0D + 0.2770

Reference Point (Normal Cohort): (0.7238577, 0.7763446599999999)

Final 'Distance %(1)' Threshold (k_2): 1.730454

Final Regression Difference Threshold (k_1): 0.2860
Total Positive Cancers: 33

Calculating thresholds for distance_threshold_offset = 1.0

Regression Equation: Avg_Death_0D = 0.3337 * 0-Dim Hole Min Life Range + 0.0361 * 0-Dim Hol

In [3]:
# Run the function/module get_thresholds_2 to determine the cancer detection 
# thresholds k_3 and k_4 for the selected biomarker set of the dataset by using the local calibration sample

import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression

def get_thresholds_2(feature_vector_path, random_patient_range, control_data_range, 
                       regression_threshold, distance_threshold, tumor_types, 
                       normal_1D_hole_max_life_range, normal_positive_required=36):
    # Step 1: Load the pre-calculated feature vector data
    new_data = pd.read_csv(feature_vector_path)
    
    # Step 2: Train Linear Regression Model using control data
    control_data = new_data.iloc[:control_data_range]

    X = control_data[['0-Dim Hole Min Life Range', '0-Dim Hole Max Life Range', 'Median_Death_0D', 'Std_Death_0D']]
    y = control_data['Avg_Death_0D']
    model = LinearRegression().fit(X, y)

    # Step 3: Predict and Analyze
    def predict_Avg_Death_0D(row):
        return np.dot(model.coef_, [row[col] for col in X.columns]) + model.intercept_

    new_data['Predicted Avg_Death_0D'] = new_data.apply(predict_Avg_Death_0D, axis=1)
    new_data['Difference'] = new_data['Avg_Death_0D'] - new_data['Predicted Avg_Death_0D']
    new_data['Regression Difference Percentage'] = abs(new_data['Difference'] / new_data['Avg_Death_0D']) * 100

    reference_point = (control_data['Average Birth (1D)'].mean(), control_data['Average Death (1D)'].mean())

    new_data['Distance %(1)'] = new_data.apply(
        lambda row: np.sqrt((row['Average Birth (1D)'] - reference_point[0])**2 + 
                    (row['Average Death (1D)'] - reference_point[1])**2) * 100,
        axis=1
    )

    # Initialize thresholds
    adjustment = 0.001
    k_3 = normal_1D_hole_max_life_range + adjustment
    k_4 = normal_1D_hole_max_life_range - adjustment
    
    max_iterations = 100
    found_thresholds = False

    for iteration in range(max_iterations):
        # Apply current thresholds
        new_data['Result(Cancer)'] = new_data.apply(
            lambda row: 'Positive' if sum([
                row['Regression Difference Percentage'] > regression_threshold,
                row['Distance %(1)'] > distance_threshold,
                row['1-Dim Hole Max Life Range'] > k_3,
                row['1-Dim Hole Max Life Range'] < k_4
            ]) >= 2 else 'Negative',
            axis=1
        )

        # Count positive normals
        positive_counts = new_data[new_data['Result(Cancer)'] == 'Positive'].groupby('Cohort').size()
        current_normal_positive = positive_counts.get('Normal', 0)
        
        print(f"Iteration {iteration}: k_3={k_3:.4f}, k_4={k_4:.4f}, Normal Positive={current_normal_positive}")
        
        # Check if we've reached the target
        if current_normal_positive <= normal_positive_required:
            found_thresholds = True
            break
            
        # If not, increase adjustment step and update thresholds
        adjustment += 0.001
        k_3 = normal_1D_hole_max_life_range + adjustment
        k_4 = normal_1D_hole_max_life_range - adjustment

    if not found_thresholds:
        print("Warning: Failed to find exact thresholds within maximum iterations")

    # Final calculation with found thresholds
    new_data['Result(Cancer)'] = new_data.apply(
        lambda row: 'Positive' if sum([
            row['Regression Difference Percentage'] > regression_threshold,
            row['Distance %(1)'] > distance_threshold,
            row['1-Dim Hole Max Life Range'] > k_3,
            row['1-Dim Hole Max Life Range'] < k_4
        ]) >= 2 else 'Negative',
        axis=1
    )

    # Step 4: Count positive results per cohort
    positive_counts = new_data[new_data['Result(Cancer)'] == 'Positive'].groupby('Cohort').size()

    # Filter the results based on user-defined cancer types
    results = {}
    for tumor_type in tumor_types:
        results[f"{tumor_type} Positive"] = positive_counts.get(tumor_type, 0)

    # Calculate total positive cancers excluding 'Normal'
    results["Total Positive Cancers"] = sum(positive_counts.values) - positive_counts.get('Normal', 0)
    results["Final k_3"] = k_3
    results["Final k_4"] = k_4
    results["Normal Positive Count"] = positive_counts.get('Normal', 0)
    results["Final Adjustment"] = adjustment

    return results

# Example usage: (Provide the required user-defined values)
feature_vector_path = "D:/Cancer Detection folder/Biomarker Selection/FeatureVectorCancerDet.csv"
random_patient_range = 322
control_data_range = 50
regression_threshold = 0.286
distance_threshold = 1.63
tumor_types = ['Bladder', 'Ovarian', 'Pancreatic']
normal_1D_hole_max_life_range = 0.1026

results = get_thresholds_2(
    feature_vector_path=feature_vector_path,
    random_patient_range=random_patient_range,
    control_data_range=control_data_range,
    regression_threshold=regression_threshold,
    distance_threshold=distance_threshold,
    tumor_types=tumor_types,
    normal_1D_hole_max_life_range=normal_1D_hole_max_life_range,
    normal_positive_required=3
)

print(results)

Iteration 0: k_3=0.1036, k_4=0.1016, Normal Positive=17
Iteration 1: k_3=0.1046, k_4=0.1006, Normal Positive=15
Iteration 2: k_3=0.1056, k_4=0.0996, Normal Positive=13
Iteration 3: k_3=0.1066, k_4=0.0986, Normal Positive=10
Iteration 4: k_3=0.1076, k_4=0.0976, Normal Positive=9
Iteration 5: k_3=0.1086, k_4=0.0966, Normal Positive=8
Iteration 6: k_3=0.1096, k_4=0.0956, Normal Positive=7
Iteration 7: k_3=0.1106, k_4=0.0946, Normal Positive=7
Iteration 8: k_3=0.1116, k_4=0.0936, Normal Positive=7
Iteration 9: k_3=0.1126, k_4=0.0926, Normal Positive=6
Iteration 10: k_3=0.1136, k_4=0.0916, Normal Positive=6
Iteration 11: k_3=0.1146, k_4=0.0906, Normal Positive=4
Iteration 12: k_3=0.1156, k_4=0.0896, Normal Positive=4
Iteration 13: k_3=0.1166, k_4=0.0886, Normal Positive=4
Iteration 14: k_3=0.1176, k_4=0.0876, Normal Positive=4
Iteration 15: k_3=0.1186, k_4=0.0866, Normal Positive=3
{'Bladder Positive': 9, 'Ovarian Positive': 11, 'Pancreatic Positive': 17, 'Total Positive Cancers': 37, 'Fina

In [4]:
# Cancer Detection Module
# Run the function/module TDAcancer_detect_1 to detect cancer for the patients
# across the entire dataset.

import pandas as pd
import numpy as np
import os
import gudhi
from sklearn.linear_model import LinearRegression

def TDAcancer_detect_1(file_path, biomarkers, random_patient_range, control_data_range, regression_threshold, distance_threshold, max_life_range_threshold_upper, max_life_range_threshold_lower, tumor_types):
    # Step 1: Load the Excel data
    df_1 = pd.read_excel(file_path, sheet_name=sheet_name)
    df_2 = pd.read_excel(file_path, sheet_name=sheet_name)

    # Step 2: Select the biomarkers (user-defined)
    df_1 = df_1[biomarkers].iloc[:59]  # Fixed dataset of 59 normal patients
    df_2 = df_2[biomarkers]

    # Step 3: Initialize list for topological features
    data_list = []

    # Step 4: Loop through rows 60 to random_patient_range and perform TDA analysis
    for index in range(60, random_patient_range + 1):
        row_to_add = df_2.iloc[[index]]
        df_new = pd.concat([df_1, row_to_add], ignore_index=True)

        # Log normalize numeric columns
        numeric_cols = df_new.select_dtypes(include=[np.number]).columns
        df_new[numeric_cols] = np.log(df_new[numeric_cols])

        # Pearson correlation matrix
        corr_matrix = np.corrcoef(df_new.values, rowvar=False)
        distance_matrix = 1 - np.abs(pd.DataFrame(corr_matrix).values)

        # Compute persistence homology
        rips_complex = gudhi.RipsComplex(distance_matrix=distance_matrix)
        simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
        persistence = simplex_tree.persistence()

        # Extract 1D and 0D hole statistics
        finite_persistence = [(dim, (birth, death)) for dim, (birth, death) in persistence if death != float('inf')]
        one_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 1]
        zero_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 0]

        max_life_range_1D = max((death - birth for birth, death in one_dim_holes), default=0)
        avg_birth_1D = np.mean([birth for birth, _ in one_dim_holes]) if one_dim_holes else 0
        avg_death_1D = np.mean([death for _, death in one_dim_holes]) if one_dim_holes else 0
        num_one_dim_holes = len(one_dim_holes)

        min_life_range_0D = min((death - birth for birth, death in zero_dim_holes), default=0)
        max_life_range_0D = max((death - birth for birth, death in zero_dim_holes), default=0)
        avg_death_0D = np.mean([death for _, death in zero_dim_holes]) if zero_dim_holes else 0
        num_zero_dim_holes = len(zero_dim_holes) + 1
        range_death_0D = max([death for _, death in zero_dim_holes], default=0) - min([death for _, death in zero_dim_holes], default=0)
        median_death_0D = np.median([death for _, death in zero_dim_holes]) if zero_dim_holes else 0
        std_death_0D = np.std([death for _, death in zero_dim_holes]) if zero_dim_holes else 0

        # Store feature vector
        data = {
            "0-Dim Hole Min Life Range": [min_life_range_0D],
            "0-Dim Hole Max Life Range": [max_life_range_0D],
            "Range_Death_0D": [range_death_0D],
            "Num_0D_Holes": [num_zero_dim_holes],
            "Median_Death_0D": [median_death_0D],
            "Std_Death_0D": [std_death_0D],
            "Avg_Death_0D": [avg_death_0D],
            "Average Birth (1D)": [avg_birth_1D],
            "Average Death (1D)": [avg_death_1D],
            "1-Dim Hole Max Life Range": [max_life_range_1D],
            "Number of 1-Dimensional Holes": [num_one_dim_holes]
        }
        df = pd.DataFrame(data, index=[f"P_{index}"])
        data_list.append(df)

    # Step 5: Save results to CSV
    final_df = pd.concat(data_list)

    # Add Patient ID, Sample ID, Cohort, and AJCC Stage to the final CSV
    additional_columns = ['Patient ID #', 'Cohort', 'AJCC Stage']
    df_clinical = pd.read_excel(file_path, sheet_name=sheet_name)
    df_additional_info = df_clinical[additional_columns].iloc[60:random_patient_range + 1]  # Corresponding rows to additional patients
    additional_info_index = [f"P_{i}" for i in range(60, random_patient_range + 1)]
    df_additional_info.index = additional_info_index

    # Merge the additional columns with the TDA results
    df_combined = pd.concat([df_additional_info, final_df], axis=1)

    # Get directory from original file path and create consistent output filename
    directory = os.path.dirname(file_path)
    csv_file_path = os.path.join(directory, "Clinical cancer data_features.csv")
    df_combined.to_csv(csv_file_path, index=True)

    # Step 6: Train Linear Regression Model
    new_data = pd.read_csv(csv_file_path)
    control_data = new_data.iloc[:control_data_range]  # User-defined control data range

    X = control_data[['0-Dim Hole Min Life Range', '0-Dim Hole Max Life Range', 'Median_Death_0D', 'Std_Death_0D']]
    y = control_data['Avg_Death_0D']
    model = LinearRegression().fit(X, y)

    # Step 7: Predict and Analyze
    def predict_Avg_Death_0D(row):
        return np.dot(model.coef_, [row[col] for col in X.columns]) + model.intercept_

    new_data['Predicted Avg_Death_0D'] = new_data.apply(predict_Avg_Death_0D, axis=1)
    new_data['Difference'] = new_data['Avg_Death_0D'] - new_data['Predicted Avg_Death_0D']
    new_data['Regression Difference Percentage'] = abs(new_data['Difference'] / new_data['Avg_Death_0D']) * 100

    reference_point = (control_data['Average Birth (1D)'].mean(), control_data['Average Death (1D)'].mean())

    new_data['Distance %(1)'] = new_data.apply(
        lambda row: np.sqrt((row['Average Birth (1D)'] - reference_point[0]) ** 2 + 
                    (row['Average Death (1D)'] - reference_point[1]) ** 2) * 100,
        axis=1
    )

    new_data['Result(Cancer)'] = new_data.apply(
        lambda row: 'Positive' if sum([
            row['Regression Difference Percentage'] > k_1,
            row['Distance %(1)'] > k_2,
            row['1-Dim Hole Max Life Range'] > k_3,
            row['1-Dim Hole Max Life Range'] < k_4
        ]) >= 2 else 'Negative',
        axis=1
    )

    new_data.to_csv(csv_file_path, index=False)

    # Step 8: Count positive results per cohort
    positive_counts = new_data[new_data['Result(Cancer)'] == 'Positive'].groupby('Cohort').size()

    # Filter the results based on user-defined cancer types
    results = {}
    for tumor_type in tumor_types:
        results[f"{tumor_type} Positive"] = positive_counts.get(tumor_type, 0)

    # Calculate total positive cancers excluding 'Normal'
    results["Total Positive Cancers"] = sum(positive_counts.values) - positive_counts.get('Normal', 0)

    return results

# Example usage: (Provide the required user-defined values)
file_path = r'D:\Cancer Detection folder\Test 3\Clinical cancer validation data.xlsx'
sheet_name = 'Normal and Cancer'
# User-defined variables
biomarkers = ['sc-kit/SCFR', 'CA-125', 'CA 19-9', 'HE4', 'Prolactin', 'SCF', 'Total PSA', 'IGFBP3']
random_patient_range = 322  
control_data_range = 50  
k_1 = 0.286  
k_2 = 1.63  
k_3 = 0.1186 
k_4 = 0.0866 
tumor_types = ['Normal', 'Bladder', 'Ovarian', 'Pancreatic'] 

# Run the pipeline
results = TDAcancer_detect_1(
    file_path, biomarkers, random_patient_range, control_data_range, 
    k_1, k_2, k_3, k_4, tumor_types
)
print(results)

{'Normal Positive': 8, 'Bladder Positive': 26, 'Ovarian Positive': 32, 'Pancreatic Positive': 39, 'Total Positive Cancers': 97}


In [5]:
# Tissue of Origin Localization Module
# Run the function/module TDAcancer_detect_2_LC to evaluate tissue localization 
# performance in the local calibration sample for different biomarker combinations.

import os
import re
import pandas as pd
import numpy as np
import gudhi
from itertools import combinations
import shutil

def TDAcancer_detect_2_LC(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range, cohorts_to_analyze):
    
    # ==================================================
    # Part 1: Preliminary Analysis and Formation of Average1.xlsx
    # ==================================================

    def calculate_averages(output_dir, tumor_types, total_positive_patients, num_combinations):
        # File paths
        cancer_result_file = os.path.join(output_dir, 'Clinical cancer data_features.csv')
        base_feature_vector_path = os.path.join(output_dir, 'FeatureVectorTissueDet{}.csv')
        base_average_path = os.path.join(output_dir, 'AverageDet{}.csv')

        # Load the cancer results file
        cancer_data = pd.read_csv(cancer_result_file)

        # Columns for averaging
        columns_to_average = ['0-Dim Hole Min Life Range', 'Range_Death_0D']

        # Process files from FeatureVectorTissueDet1.csv to FeatureVectorTissueDetN.csv
        for i in range(1, num_combinations + 1):
            file_path = base_feature_vector_path.format(i)
            try:
                new_data = pd.read_csv(file_path)
            except FileNotFoundError:
                print(f"File {file_path} not found, skipping.")
                continue
            
            # Add 'Result(Cancer)' column
            new_data['Result(Cancer)'] = cancer_data['Result(Cancer)']
            new_data.to_csv(file_path, index=False)
            
            # Convert numeric columns to numeric type
            for col in columns_to_average:
                new_data[col] = pd.to_numeric(new_data[col], errors='coerce')
            
            # Compute cohort averages - ONLY USE THE SPECIFIED NUMBER OF PATIENTS
            averages_list = []
            for tumor, row_limit in tumor_types.items():
                if tumor == 'normal':
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Negative')]
                else:
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Positive')]
                
                # Use only the specified number of patients for each cohort
                selected_data = filtered_data.iloc[:row_limit][columns_to_average].dropna()
                # Convert to numeric and calculate mean
                numeric_means = selected_data.astype(float).mean().to_dict()
                averages_list.append({'Cohort': tumor.capitalize(), **numeric_means})
            
            averages_data = pd.DataFrame(averages_list)
            normal_averages = averages_data[averages_data['Cohort'].str.lower() == 'normal']
            reference_point = (
                float(normal_averages['0-Dim Hole Min Life Range'].values[0]),
                float(normal_averages['Range_Death_0D'].values[0])
            )
            
            # Compute Euclidean distance
            X = new_data[['0-Dim Hole Min Life Range']].values.astype(float)
            y = new_data['Range_Death_0D'].values.astype(float)
            distances = [np.sqrt((X[i][0] - reference_point[0]) ** 2 + (y[i] - reference_point[1]) ** 2) for i in range(len(X))]
            new_data['Distance %(3)'] = [d * 100 for d in distances]
            new_data.to_csv(file_path, index=False)
            
            # Compute distance averages - ONLY USE THE SPECIFIED NUMBER OF PATIENTS
            averages_distance = []
            for tumor, row_limit in tumor_types.items():
                if tumor == 'normal':
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Negative')]
                else:
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Positive')]
                
                # Use only the specified number of patients for each cohort
                selected_data = filtered_data.iloc[:row_limit][columns_to_average + ['Distance %(3)']].dropna()
                # Convert to numeric and calculate mean
                numeric_means = selected_data.astype(float).mean().to_dict()
                averages_distance.append({'Cohort': tumor.capitalize(), 'Average Distance %(3)': numeric_means.get('Distance %(3)', 0), **numeric_means})
            
            averages_distance_data = pd.DataFrame(averages_distance)
            averages_distance_data.to_csv(base_average_path.format(i), index=False)
            
            # Determine threshold
            sorted_distances = averages_distance_data['Average Distance %(3)'].sort_values(ascending=False)
            second_highest_distance = sorted_distances.iloc[1]
            k = int(np.ceil(second_highest_distance))
            
            # Store k value for later use in offset calculations
            new_data['k_value'] = k
            new_data.to_csv(file_path, index=False)
            
            print(f"Processed and updated {file_path} and {base_average_path.format(i)} successfully.")

    def merge_averages_to_excel(output_dir, num_combinations):
        # Function for natural sorting (e.g., 1,2,3 instead of 1,10,100)
        def natural_sort_key(text):
            return [int(num) if num.isdigit() else num for num in re.split(r'(\d+)', text)]

        # Merge AverageDet CSV files into Average1.xlsx
        # Get a sorted list of all relevant CSV files
        csv_files = sorted(
            [f for f in os.listdir(output_dir) if f.startswith("AverageDet") and f.endswith(".csv")],
            key=natural_sort_key
        )

        # Initialize an empty DataFrame for merging
        merged_df = None

        # Loop through each file and process
        for i, file in enumerate(csv_files, start=1):
            if i > num_combinations:
                break  # Stop after processing the specified number of combinations
            
            # Read the current CSV file
            file_path = os.path.join(output_dir, file)
            df = pd.read_csv(file_path)

            # Select only the "Cohort" and "Average Distance %(3)" columns
            df = df[["Cohort", "Average Distance %(3)"]]

            # Rename the "Average Distance %(3)" column to include the file index
            df.rename(columns={"Average Distance %(3)": f"Average Distance %(3)-{i}"}, inplace=True)

            # Merge with accumulated DataFrame
            if merged_df is None:
                merged_df = df
            else:
                merged_df = pd.merge(merged_df, df, on="Cohort", how="outer")

        # Save the merged DataFrame to an Excel file
        output_file_path = os.path.join(output_dir, "Average1.xlsx")
        merged_df.to_excel(output_file_path, index=False, engine='openpyxl')
        print(f"Merged file saved as: {output_file_path}")

    def process_clinical_data(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range):
        # Step 1: Load the clinical data
        df_1 = pd.read_excel(file_path, sheet_name=sheet_name)
        df_2 = pd.read_excel(file_path, sheet_name=sheet_name)

        # Generate all combinations of 3 biomarkers
        biomarker_combinations = list(combinations(biomarkers, 3))
        if num_combinations > len(biomarker_combinations):
            print(f"Warning: Only {len(biomarker_combinations)} combinations available. Processing {len(biomarker_combinations)} combinations instead of {num_combinations}.")
            num_combinations = len(biomarker_combinations)

        # Directory to save the CSV files
        os.makedirs(output_dir, exist_ok=True)

        # Step 2: Iterate through each combination of biomarkers
        for combo_index, combo in enumerate(biomarker_combinations[:num_combinations], start=1):
            # Select biomarkers for this iteration
            columns_to_select = list(combo)
            print(f"Processing combination {combo_index}: {columns_to_select}")

            # Subset the data
            df_1_subset = df_1[columns_to_select].iloc[:59]  # First 59 rows for normal patients
            df_2_subset = df_2[columns_to_select]           # All rows for cancer patients

            # Initialize a list to store feature vectors
            data_list = []

            # Step 4: Add additional random patient data for analysis
            for index in range(60, random_patient_range + 1):  # User-defined range
                # Select a specific row from df_2
                row_to_add = df_2_subset.iloc[[index]]

                # Combine it with the fixed dataset
                df_combined = pd.concat([df_1_subset] + [row_to_add] * 1, ignore_index=True)

                # Step 5: Log normalize the combined data
                numeric_cols = df_combined.select_dtypes(include=[np.number]).columns
                df_combined[numeric_cols] = np.log(df_combined[numeric_cols])

                # Convert DataFrame to numpy array
                df_subset = df_combined.values

                # Step 6: Calculate Pearson correlation matrix and distance matrix
                corr_matrix = np.corrcoef(df_subset, rowvar=False)
                corr_matrix_df = pd.DataFrame(corr_matrix, columns=columns_to_select, index=columns_to_select)
                distance_matrix = 1 - np.abs(corr_matrix_df)
                distance_matrix_np = distance_matrix.values

                # Step 7: Construct a simplicial complex using Gudhi's RipsComplex
                rips_complex = gudhi.RipsComplex(distance_matrix=distance_matrix_np)
                simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
                persistence = simplex_tree.persistence()

                # Step 8: Extract topological features
                finite_persistence = [(dim, (birth, death)) for dim, (birth, death) in persistence if death != float('inf')]
                one_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 1]
                
                if one_dim_holes:
                    max_life_range_1D = max(death - birth for birth, death in one_dim_holes)
                    avg_birth_1D = np.mean([birth for birth, _ in one_dim_holes])
                    avg_death_1D = np.mean([death for _, death in one_dim_holes])
                    num_one_dim_holes = len(one_dim_holes)
                else:
                    max_life_range_1D = avg_birth_1D = avg_death_1D = num_one_dim_holes = 0

                zero_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 0]
                min_life_range_0D = min(death - birth for birth, death in zero_dim_holes)
                max_life_range_0D = max(death - birth for birth, death in zero_dim_holes)
                avg_death_0D = np.mean([death for _, death in zero_dim_holes])
                num_zero_dim_holes = len(zero_dim_holes) + 1
                range_death_0D = max([death for _, death in zero_dim_holes]) - min([death for _, death in zero_dim_holes])
                median_death_0D = np.median([death for _, death in zero_dim_holes])
                std_death_0D = np.std([death for _, death in zero_dim_holes])

                # Prepare feature vector
                data = {
                    "0-Dim Hole Min Life Range": [min_life_range_0D],
                    "0-Dim Hole Max Life Range": [max_life_range_0D],
                    "Range_Death_0D": [range_death_0D],
                    "Num_0D_Holes": [num_zero_dim_holes],
                    "Median_Death_0D": [median_death_0D],
                    "Std_Death_0D": [std_death_0D],
                    "Avg_Death_0D": [avg_death_0D],
                    "Average Birth (1D)": [avg_birth_1D],
                    "Average Death (1D)": [avg_death_1D],
                    "1-Dim Hole Max Life Range": [max_life_range_1D],
                    "Number of 1-Dimensional Holes": [num_one_dim_holes]
                }

                # Append to list
                df_features = pd.DataFrame(data, index=[f"P_{index}"])
                data_list.append(df_features)

            # Concatenate all feature vectors for the current combination
            final_df = pd.concat(data_list)

            # Add additional clinical information
            additional_columns = ['Patient ID #', 'Cohort', 'AJCC Stage']
            df_clinical = pd.read_excel(file_path, sheet_name=sheet_name)
            df_additional_info = df_clinical[additional_columns].iloc[60:random_patient_range + 1]
            additional_info_index = [f"P_{i}" for i in range(60, random_patient_range + 1)]
            df_additional_info.index = additional_info_index

            # Merge with topological data
            df_combined = pd.concat([df_additional_info, final_df], axis=1)

            # Append biomarkers as the last row
            selected_biomarkers_row = pd.DataFrame(
                {col: None for col in df_combined.columns}, 
                index=["Selected Biomarkers"]
            )
            selected_biomarkers_row.iloc[0, :len(columns_to_select)] = columns_to_select
            df_combined = pd.concat([df_combined, selected_biomarkers_row])

            # Save to a CSV file
            csv_file_path = os.path.join(output_dir, f"FeatureVectorTissueDet{combo_index}.csv")
            df_combined.to_csv(csv_file_path, float_format="%f", index=True)
            print(f"Saved: {csv_file_path}")

        # Step 9: Calculate averages and generate Average1.xlsx
        calculate_averages(output_dir, tumor_types, total_positive_patients, num_combinations)
        merge_averages_to_excel(output_dir, num_combinations)

    # ==================================================
    # Part 2: Formation of Positives1.xlsx and Sensitivity Outcomes
    # ==================================================

    def apply_offset_and_create_positives(output_dir, tumor_types_dict, num_combinations):
        # Create offset directories and apply different thresholds
        for offset_index, offset in enumerate(range(-20, 20), start=1):
            offset_dir = os.path.join(output_dir, f'Positives{offset_index}')
            os.makedirs(offset_dir, exist_ok=True)
            
            print(f"Processing offset {offset} (index {offset_index})")
            
            # Copy the base files to the offset directory
            for i in range(1, num_combinations + 1):
                src_file = os.path.join(output_dir, f'FeatureVectorTissueDet{i}.csv')
                dst_file = os.path.join(offset_dir, f'FeatureVectorTissueDet{i}.csv')
                if os.path.exists(src_file):
                    shutil.copy2(src_file, dst_file)
            
            # Copy Average1.xlsx to the offset directory
            src_avg = os.path.join(output_dir, 'Average1.xlsx')
            dst_avg = os.path.join(offset_dir, 'Average1.xlsx')
            if os.path.exists(src_avg):
                shutil.copy2(src_avg, dst_avg)
            
            # Process each FeatureVector file with the offset threshold
            for i in range(1, num_combinations + 1):
                input_file_path = os.path.join(offset_dir, f'FeatureVectorTissueDet{i}.csv')
                try:
                    # Load the data
                    new_data = pd.read_csv(input_file_path)
                    
                    # Get the k value
                    k = new_data['k_value'].iloc[0]
                    
                    # Apply offset to threshold
                    threshold = k + offset
                    
                    # Localize cancer tissue with offset threshold
                    new_data['Result(Tissue Detection)'] = ['Positive' if value > threshold else 'Negative' for value in new_data['Distance %(3)']]
                    new_data['Cancer tissue'] = new_data.apply(
                        lambda row: 'Positive' if row['Result(Cancer)'] == 'Positive' and row['Result(Tissue Detection)'] == 'Positive' else 'Negative', 
                        axis=1
                    )
                    
                    # Save the updated file
                    new_data.to_csv(input_file_path, index=False)
                    
                except FileNotFoundError:
                    print(f"File not found: {input_file_path}, skipping...")
            
            # Now create positives count files for this offset
            merge_positives_to_excel(offset_dir, tumor_types_dict, num_combinations, offset_index)
            calculate_metrics(offset_dir, tumor_types_dict, total_positive_patients, num_combinations, offset_index)
            get_top_10_metrics(offset_dir, tumor_types_dict, num_combinations, offset_index)

    def merge_positives_to_excel(output_dir, tumor_types_dict, num_combinations, offset_index=None):
        # Directory paths
        input_dir = output_dir
        positives_dir = os.path.join(input_dir, 'Positives')

        # Ensure the output directory exists
        os.makedirs(positives_dir, exist_ok=True)

        # Loop through files 1 to N
        for i in range(1, num_combinations + 1):
            input_file_path = os.path.join(input_dir, f'FeatureVectorTissueDet{i}.csv')
            output_file_path = os.path.join(positives_dir, f'positives_countDet{i}.csv')

            try:
                # Step 1: Load the CSV file
                new_data = pd.read_csv(input_file_path, header=0)

                # Step 2: Initialize an empty list to store results
                results = []

                # Step 3: Count the number of positives for each Cohort and each AJCC Stage
                # USE ONLY POSITIVE PATIENTS (Result(Cancer) == 'Positive') AND THE SPECIFIED NUMBER OF PATIENTS
                for tumor_type, row_limit in tumor_types_dict.items():
                    if tumor_type == 'normal':
                        continue  # Skip normal cohort for positive counts
                    
                    # Filter data for the current Cohort - ONLY POSITIVE PATIENTS
                    tumor_data = new_data[
                        (new_data['Cohort'].str.lower() == tumor_type.lower()) & 
                        (new_data['Result(Cancer)'] == 'Positive')
                    ]
                    
                    # Use only the specified number of patients for this cohort
                    tumor_data = tumor_data.iloc[:row_limit]

                    # Count positives for each AJCC stage (I, II, III)
                    for stage in ['I', 'II', 'III']:
                        stage_data = tumor_data[tumor_data['AJCC Stage'] == stage]

                        # Count positives for 'Cancer tissue' column
                        positive_count = stage_data[stage_data['Cancer tissue'] == 'Positive'].shape[0]

                        # Append result to the list
                        results.append({'Cohort': tumor_type.capitalize(), 'AJCC Stage': stage, 'Positive Count': positive_count})

                    # Count positives for all AJCC stages combined
                    total_positive_count = tumor_data[tumor_data['Cancer tissue'] == 'Positive'].shape[0]
                    results.append({'Cohort': tumor_type.capitalize(), 'AJCC Stage': 'All Stages', 'Positive Count': total_positive_count})

                # Step 4: Convert the results into a DataFrame and save to CSV
                results_df = pd.DataFrame(results)
                results_df.to_csv(output_file_path, index=False)

                print(f"Processed: {input_file_path} → Saved: {output_file_path}")

            except FileNotFoundError:
                print(f"File not found: {input_file_path}, skipping...")

        # Merge all positives_countDet CSV files into Positives1.xlsx
        # Function for natural sorting (e.g., 1,2,3 instead of 1,10,100)
        def natural_sort_key(text):
            return [int(num) if num.isdigit() else num for num in re.split(r'(\d+)', text)]

        # Get a sorted list of all relevant CSV files
        csv_files = sorted(
            [f for f in os.listdir(positives_dir) if f.startswith("positives_countDet") and f.endswith(".csv")],
            key=natural_sort_key
        )

        # Initialize an empty list to store DataFrames
        dfs = []

        # Loop through each file and process it
        for i, file in enumerate(csv_files, start=1):
            if i > num_combinations:
                break  # Stop after processing the specified number of combinations
            
            # Read the current CSV file
            file_path = os.path.join(positives_dir, file)
            df = pd.read_csv(file_path)

            # Filter rows based on conditions - ONLY INCLUDE THE SPECIFIED COHORTS
            df = df[(df["AJCC Stage"] == "All Stages") & (df["Cohort"].isin([c.capitalize() for c in tumor_types_dict.keys() if c != 'normal']))]

            # Rename the "Positive Count" column to include the file index
            df.rename(columns={"Positive Count": f"Positive Det {i}"}, inplace=True)

            # Append the processed DataFrame to the list
            dfs.append(df)

        # Merge all DataFrames on "Cohort" and "AJCC Stage"
        if dfs:
            merged_df = dfs[0]
            for df in dfs[1:]:
                merged_df = pd.merge(merged_df, df, on=["Cohort", "AJCC Stage"], how="outer")

            # Sort the final DataFrame by "Cohort" and "AJCC Stage" for consistency
            merged_df = merged_df.sort_values(by=["Cohort", "AJCC Stage"]).reset_index(drop=True)

            # Add "Total Positive Patients" column - USE THE total_positive_patients DICTIONARY
            merged_df["Total Positive Patients"] = merged_df["Cohort"].map(total_positive_patients)

            # Reorder columns to place "Total Positive Patients" after "AJCC Stage"
            cols = ["Cohort", "AJCC Stage", "Total Positive Patients"] + [col for col in merged_df.columns if col.startswith("Positive Det")]
            merged_df = merged_df[cols]

            # Save the merged DataFrame to an Excel file
            if offset_index:
                output_file_path = os.path.join(positives_dir, f"Positives{offset_index}.xlsx")
            else:
                output_file_path = os.path.join(positives_dir, "Positives1.xlsx")
            merged_df.to_excel(output_file_path, index=False, engine='openpyxl')

            print(f"Merged file saved as: {output_file_path}")
        else:
            print("No data to merge for Positives1.xlsx")

    def calculate_metrics(output_dir, tumor_types_dict, total_positive_patients, num_combinations, offset_index=None):
        # Determine the positives file path based on offset index
        if offset_index:
            positives_file_path = os.path.join(output_dir, 'Positives', f'Positives{offset_index}.xlsx')
        else:
            positives_file_path = os.path.join(output_dir, 'Positives', 'Positives1.xlsx')
            
        averages_file_path = os.path.join(output_dir, 'Average1.xlsx')

        positives_df = pd.read_excel(positives_file_path)
        averages_df = pd.read_excel(averages_file_path)

        # Get the list of Positive Count columns
        positive_count_columns = [col for col in positives_df.columns if col.startswith("Positive Det")]

        # Initialize columns for results
        results = {"Sensitivity": [], "Precision": [], "F1_Score": [], "Accuracy": [], "Cohort": []}

        # Total number of positive patients across all Cohorts - USE THE total_positive_patients DICTIONARY
        total_patients = positives_df["Total Positive Patients"].sum()

        # Loop through each Positive Count column
        for i, column in enumerate(positive_count_columns, start=1):
            if i > num_combinations:
                break  # Stop after processing the specified number of combinations
            
            # Determine the Cohort type based on the highest value in the corresponding "Average Distance %(3)-i" column
            avg_column = f"Average Distance %(3)-{i}"
            
            if avg_column not in averages_df.columns:
                print(f"Column {avg_column} not found in Averages file, skipping...")
                continue  # Skip this iteration if column is missing
            
            max_value_row = averages_df[avg_column].idxmax()
            Cohort_type = averages_df.loc[max_value_row, "Cohort"]

            # Ensure the Cohort exists in positives_df
            Cohort_row = positives_df[positives_df["Cohort"] == Cohort_type]
            if Cohort_row.empty:
                print(f"Cohort {Cohort_type} not found in Positives file, skipping...")
                continue
            
            Cohort_row = Cohort_row.iloc[0]  # Convert to Series
            
            # Extract necessary values
            TP = Cohort_row[column]  # True Positives
            FN = Cohort_row["Total Positive Patients"] - TP  # False Negatives
            
            # Calculate FP and TN using the total_positive_patients dictionary
            # For FP: Sum of all positives detected minus TP for this cohort
            FP = positives_df[column].sum() - TP
            
            # For TN: Total patients minus (TP + FP + FN)
            # Total patients is the sum of all values in total_positive_patients
            TN = total_patients - (TP + FP + FN)

            # Calculate Sensitivity, Precision, F1 Score, and Accuracy
            sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            f1_score = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
            accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
            
            # Append results
            results["Sensitivity"].append(sensitivity)
            results["Precision"].append(precision)
            results["F1_Score"].append(f1_score)
            results["Accuracy"].append(accuracy)
            results["Cohort"].append(Cohort_type)

            # Write results into the DataFrame
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Sensitivity {i}"] = sensitivity
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Precision {i}"] = precision
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"F1 Score {i}"] = f1_score
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Accuracy {i}"] = accuracy

        # Save the results back to the Excel file
        positives_df.to_excel(positives_file_path, index=False, engine='openpyxl')

        print(f"Results saved to: {positives_file_path}")

    def get_top_10_metrics(output_dir, tumor_types_dict, num_combinations, offset_index=None):
        if offset_index:
            excel_file_path = os.path.join(output_dir, 'Positives', f'Positives{offset_index}.xlsx')
        else:
            excel_file_path = os.path.join(output_dir, 'Positives', 'Positives1.xlsx')
            
        csv_file_folder = output_dir
        output_path = os.path.join(output_dir, 'Positives')

        # Read the Excel file
        excel_df = pd.read_excel(excel_file_path)

        # Function for natural sorting (e.g., 1,2,3 instead of 1,10,100)
        def natural_sort_key(text):
            return [int(num) if num.isdigit() else num for num in re.split(r'(\d+)', text)]

        # Function to get selected biomarkers from the appropriate CSV file
        def get_selected_biomarkers(csv_file_name, col_start=1, col_end=4):
            csv_file_path = os.path.join(csv_file_folder, csv_file_name)
            if os.path.exists(csv_file_path):
                csv_df = pd.read_csv(csv_file_path)
                return csv_df.iloc[-1, col_start:col_end].values.tolist()  # Convert NumPy array to Python list
            else:
                print(f"Warning: File {csv_file_name} not found.")
                return ["NA"]  # Return "NA" instead of an empty list

        # Initialize a list to store results
        results = []

        # Process each Cohort (row) in the Excel file
        for idx, row in excel_df.iterrows():
            Cohort = row['Cohort']
            print(f"Processing Cohort: {Cohort}")
            
            # Initialize the list for this Cohort
            Cohort_results = {
                'Cohort': Cohort,
                'Offset': offset_index - 20 if offset_index else 0  # Convert to actual offset value (-20 to +20)
            }
            
            # Add columns for top 10 results
            for i in range(1, 11):
                Cohort_results[f'Top {i} F1 Score'] = 'NA'
                Cohort_results[f'Top {i} Accuracy'] = 'NA'
                Cohort_results[f'Top {i} Precision'] = 'NA'
                Cohort_results[f'Top {i} Sensitivity'] = 'NA'
                Cohort_results[f'Top {i} Biomarkers'] = 'NA'
            
            # Generate ordered metric column groups
            metric_groups = []
            for i in range(1, num_combinations + 1):
                metric_groups.append({
                    'f1_score': f'F1 Score {i}',
                    'accuracy': f'Accuracy {i}',
                    'precision': f'Precision {i}',
                    'sensitivity': f'Sensitivity {i}'
                })
            
            # Store valid metric groups
            valid_metrics = []
            
            for group in metric_groups:
                # Check if all columns exist in the row
                if not all(col in row for col in group.values()):
                    continue
                
                f1_score_value = row[group['f1_score']]
                accuracy_value = row[group['accuracy']]
                precision_value = row[group['precision']]
                sensitivity_value = row[group['sensitivity']]
                
                if pd.notna(f1_score_value) and pd.notna(accuracy_value) and f1_score_value > 0 and accuracy_value > 0:
                    # Determine which CSV file to read based on the metric number
                    metric_num = int(group['f1_score'].split(' ')[2])
                    csv_file_name = f'FeatureVectorTissueDet{metric_num}.csv'
                    
                    # Get selected biomarkers for the current metric group
                    biomarkers = get_selected_biomarkers(csv_file_name)
                    
                    # Store the result
                    valid_metrics.append({
                        'F1_Score': f1_score_value,
                        'Accuracy': accuracy_value,
                        'Precision': precision_value,
                        'Sensitivity': sensitivity_value,
                        'Biomarkers': biomarkers,
                        'Metric_Number': metric_num
                    })
            
            # If valid metrics exist, get the top 10 sorted by highest F1 Score (primary) and Accuracy (secondary)
            if valid_metrics:
                sorted_metrics = sorted(valid_metrics, key=lambda x: (x['F1_Score'], x['Accuracy']), reverse=True)[:10]
                
                # Update the Cohort result with the top 10 metrics
                for i, res in enumerate(sorted_metrics):
                    Cohort_results[f'Top {i+1} F1 Score'] = res['F1_Score']
                    Cohort_results[f'Top {i+1} Accuracy'] = res['Accuracy']
                    Cohort_results[f'Top {i+1} Precision'] = res['Precision']
                    Cohort_results[f'Top {i+1} Sensitivity'] = res['Sensitivity']
                    Cohort_results[f'Top {i+1} Biomarkers'] = res['Biomarkers']
            
            # Append the result for the current Cohort
            results.append(Cohort_results)

        # Create a DataFrame for the results
        results_df = pd.DataFrame(results)

        # Ensure output path exists
        os.makedirs(output_path, exist_ok=True)

        # Define output file path
        if offset_index:
            output_file_path = os.path.join(output_path, f'Top_10_Metrics_{offset_index}.xlsx')
        else:
            output_file_path = os.path.join(output_path, 'Top_10_Metrics.xlsx')

        # Save the results to Excel
        results_df.to_excel(output_file_path, index=False, engine='openpyxl')

        print(f"Results saved to {output_file_path}")

    # ==================================================
    # Part 3: Find Top 10 F1 Scores Across All Offsets
    # ==================================================

    def find_top_10_f1_scores_across_offsets(output_dir, tumor_types_dict):
        """
        Find the top 6 F1 scores across all offset files and provide the respective
        biomarker combinations and offset information for each cohort.
        """
        all_results = []
        
        # Get all Top_10_Metrics files
        metrics_files = []
        for offset_index in range(1, 41):  # Offsets from -20 to +19 (40 files)
            file_path = os.path.join(output_dir, f'Positives{offset_index}', 'Positives', f'Top_10_Metrics_{offset_index}.xlsx')
            if os.path.exists(file_path):
                actual_offset = offset_index - 21  # Convert to actual offset value (-20 to +19)
                metrics_files.append((actual_offset, file_path))
        
        # Process each cohort
        for cohort in tumor_types_dict.keys():
            if cohort == 'normal':
                continue
                
            cohort_results = []
            
            for offset, file_path in metrics_files:
                try:
                    df = pd.read_excel(file_path)
                    cohort_data = df[df['Cohort'] == cohort.capitalize()]
                    
                    if not cohort_data.empty:
                        # Get all top 10 results for this cohort and offset
                        for i in range(1, 11):
                            f1_score = cohort_data[f'Top {i} F1 Score'].values[0]
                            if pd.notna(f1_score) and f1_score != 'NA':
                                cohort_results.append({
                                    'Cohort': cohort.capitalize(),
                                    'F1_Score': f1_score,
                                    'Accuracy': cohort_data[f'Top {i} Accuracy'].values[0],
                                    'Precision': cohort_data[f'Top {i} Precision'].values[0],
                                    'Sensitivity': cohort_data[f'Top {i} Sensitivity'].values[0],
                                    'Biomarkers': cohort_data[f'Top {i} Biomarkers'].values[0],
                                    'Offset': offset,
                                    'Rank': i
                                })
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")
                    continue
            
            # Sort all results for this cohort by F1 score (descending)
            if cohort_results:
                cohort_results_sorted = sorted(cohort_results, key=lambda x: x['F1_Score'], reverse=True)[:10]
                all_results.extend(cohort_results_sorted)
        
        # Create DataFrame and save to Excel
        if all_results:
            results_df = pd.DataFrame(all_results)
            output_file = os.path.join(output_dir, 'Top_10_F1_ScoresBased_LocPerformance.xlsx')
            results_df.to_excel(output_file, index=False, engine='openpyxl')
            print(f"Top 10 F1 scores across offsets saved to: {output_file}")
            
            # Print results
            print("\n=== TOP 10 F1 SCORES ACROSS ALL OFFSETS AND TISSUE LOCALIZATION PERFORMANCE ===")
            for cohort in tumor_types_dict.keys():
                if cohort == 'normal':
                    continue
                    
                cohort_data = results_df[results_df['Cohort'] == cohort.capitalize()]
                if not cohort_data.empty:
                    print(f"\nCohort: {cohort.capitalize()}")
                    print("-" * 80)
                    for idx, row in cohort_data.iterrows():
                        print(f"Rank: {row['Rank']}")
                        print(f"F1 Score: {row['F1_Score']:.6f}")
                        print(f"Accuracy: {row['Accuracy']:.6f}")
                        print(f"Precision: {row['Precision']:.6f}")
                        print(f"Sensitivity: {row['Sensitivity']:.6f}")
                        print(f"Biomarkers: {row['Biomarkers']}")
                        print(f"Offset: {row['Offset']}")
                        print("-" * 40)
        else:
            print("No F1 scores found across offsets.")
    
    # ==================================================

    # Main Execution
    # ==================================================

    # Part 1: Preliminary Analysis and Formation of Average1.xlsx (only once)
    process_clinical_data(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range)

    # Part 2: Formation of Positives1.xlsx and Metrics Outcomes with offset thresholds
    apply_offset_and_create_positives(output_dir, tumor_types, num_combinations)
    
    # Part 3: Find top 6 F1 scores across all offsets
    find_top_10_f1_scores_across_offsets(output_dir, tumor_types)

# Example usage (Provide the required user-defined values)
file_path = r'D:\Cancer Detection folder\Test 3\Clinical cancer validation data.xlsx'
sheet_name = 'Normal and Cancer'
biomarkers = ['sc-kit/SCFR', 'CA-125', 'CA 19-9', 'HE4', 'Prolactin', 'SCF', 'Total PSA', 'IGFBP3']
output_dir = r"D:\Cancer Detection folder\Test 3"

tumor_types = {
    'normal': 47,
    'bladder': 9,
    'ovarian': 11,
    'pancreatic': 17
}
total_positive_patients = {
    "Bladder": 9,      
    "Ovarian": 11,     
    "Pancreatic": 17   
}
cohorts_to_analyze = ['Bladder', 'Ovarian', 'Pancreatic']
num_combinations = 56 
random_patient_range = 322 

# Execute the TDA_cancerDetect function
TDAcancer_detect_2_LC(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range, cohorts_to_analyze)

Processing combination 1: ['sc-kit/SCFR', 'CA-125', 'CA 19-9']


C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\3176232211.py:257: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, selected_biomarkers_row])


Saved: D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet1.csv
Processing combination 2: ['sc-kit/SCFR', 'CA-125', 'HE4']


C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\3176232211.py:257: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, selected_biomarkers_row])


Saved: D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet2.csv
Processing combination 3: ['sc-kit/SCFR', 'CA-125', 'Prolactin']


C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\3176232211.py:257: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, selected_biomarkers_row])


... (183 outputs left out)

C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\3176232211.py:513: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positives_df.loc[positives_df["Cohort"] == Cohort_type, f"F1 Score {i}"] = f1_score
C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\3176232211.py:514: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Accuracy {i}"] = accuracy
C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\3176232211.py:511: PerformanceWarning: DataFrame is high

Results saved to: D:\Cancer Detection folder\Test 3\Positives39\Positives\Positives39.xlsx
Processing Cohort: Bladder
Processing Cohort: Ovarian
Processing Cohort: Pancreatic
Results saved to D:\Cancer Detection folder\Test 3\Positives39\Positives\Top_10_Metrics_39.xlsx
Processing offset 19 (index 40)
Processed: D:\Cancer Detection folder\Test 3\Positives40\FeatureVectorTissueDet1.csv → Saved: D:\Cancer Detection folder\Test 3\Positives40\Positives\positives_countDet1.csv
Processed: D:\Cancer Detection folder\Test 3\Positives40\FeatureVectorTissueDet2.csv → Saved: D:\Cancer Detection folder\Test 3\Positives40\Positives\positives_countDet2.csv
Processed: D:\Cancer Detection folder\Test 3\Positives40\FeatureVectorTissueDet3.csv → Saved: D:\Cancer Detection folder\Test 3\Positives40\Positives\positives_countDet3.csv
Processed: D:\Cancer Detection folder\Test 3\Positives40\FeatureVectorTissueDet4.csv → Saved: D:\Cancer Detection folder\Test 3\Positives40\Positives\positives_countDet4.csv
P

C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\3176232211.py:513: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positives_df.loc[positives_df["Cohort"] == Cohort_type, f"F1 Score {i}"] = f1_score
C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\3176232211.py:514: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Accuracy {i}"] = accuracy
C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\3176232211.py:511: PerformanceWarning: DataFrame is high

Results saved to: D:\Cancer Detection folder\Test 3\Positives40\Positives\Positives40.xlsx
Processing Cohort: Bladder
Processing Cohort: Ovarian
Processing Cohort: Pancreatic
Results saved to D:\Cancer Detection folder\Test 3\Positives40\Positives\Top_10_Metrics_40.xlsx
Top 10 F1 scores across offsets saved to: D:\Cancer Detection folder\Test 3\Top_10_F1_ScoresBased_LocPerformance.xlsx

=== TOP 10 F1 SCORES ACROSS ALL OFFSETS AND TISSUE LOCALIZATION PERFORMANCE ===

Cohort: Bladder
--------------------------------------------------------------------------------
Rank: 1
F1 Score: 0.777778
Accuracy: 0.891892
Precision: 0.777778
Sensitivity: 0.777778
Biomarkers: ['sc-kit/SCFR', 'HE4', 'Total PSA']
Offset: 0
----------------------------------------
Rank: 1
F1 Score: 0.695652
Accuracy: 0.810811
Precision: 0.571429
Sensitivity: 0.888889
Biomarkers: ['sc-kit/SCFR', 'HE4', 'Total PSA']
Offset: -1
----------------------------------------
Rank: 2
F1 Score: 0.695652
Accuracy: 0.810811
Precision: 

In [6]:
# Tissue of Origin Localization Module
# Run the function/module TDAcancer_detect_2 to evaluate and validate tissue localization 
# performance in the entire dataset (local calibration + test) for different biomarker combinations.
# The performance outcomes obtained here validate the results from the local calibration samples 
# for the respective biomarker combinations and threshold offsets.

import os
import re
import pandas as pd
import numpy as np
import gudhi
from itertools import combinations
import shutil

def TDAcancer_detect_2(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range, cohorts_to_analyze):
    
    # ==================================================
    # Part 1: Preliminary Analysis and Formation of Average1.xlsx
    # ==================================================

    def calculate_averages(output_dir, tumor_types, total_positive_patients, num_combinations):
        # File paths
        cancer_result_file = os.path.join(output_dir, 'Clinical cancer data_features.csv')
        base_feature_vector_path = os.path.join(output_dir, 'FeatureVectorTissueDet{}.csv')
        base_average_path = os.path.join(output_dir, 'AverageDet{}.csv')

        # Load the cancer results file
        cancer_data = pd.read_csv(cancer_result_file)

        # Columns for averaging
        columns_to_average = ['0-Dim Hole Min Life Range', 'Range_Death_0D']

        # Process files from FeatureVectorTissueDet1.csv to FeatureVectorTissueDetN.csv
        for i in range(1, num_combinations + 1):
            file_path = base_feature_vector_path.format(i)
            try:
                new_data = pd.read_csv(file_path)
            except FileNotFoundError:
                print(f"File {file_path} not found, skipping.")
                continue
            
            # Add 'Result(Cancer)' column
            new_data['Result(Cancer)'] = cancer_data['Result(Cancer)']
            new_data.to_csv(file_path, index=False)
            
            # Convert numeric columns to numeric type
            for col in columns_to_average:
                new_data[col] = pd.to_numeric(new_data[col], errors='coerce')
            
            # Compute cohort averages - ONLY USE THE SPECIFIED NUMBER OF PATIENTS
            averages_list = []
            for tumor, row_limit in tumor_types.items():
                if tumor == 'normal':
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Negative')]
                else:
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Positive')]
                
                # Use only the specified number of patients for each cohort
                selected_data = filtered_data.iloc[:row_limit][columns_to_average].dropna()
                # Convert to numeric and calculate mean
                numeric_means = selected_data.astype(float).mean().to_dict()
                averages_list.append({'Cohort': tumor.capitalize(), **numeric_means})
            
            averages_data = pd.DataFrame(averages_list)
            normal_averages = averages_data[averages_data['Cohort'].str.lower() == 'normal']
            reference_point = (
                float(normal_averages['0-Dim Hole Min Life Range'].values[0]),
                float(normal_averages['Range_Death_0D'].values[0])
            )
            
            # Compute Euclidean distance
            X = new_data[['0-Dim Hole Min Life Range']].values.astype(float)
            y = new_data['Range_Death_0D'].values.astype(float)
            distances = [np.sqrt((X[i][0] - reference_point[0]) ** 2 + (y[i] - reference_point[1]) ** 2) for i in range(len(X))]
            new_data['Distance %(3)'] = [d * 100 for d in distances]
            new_data.to_csv(file_path, index=False)
            
            # Compute distance averages - ONLY USE THE SPECIFIED NUMBER OF PATIENTS
            averages_distance = []
            for tumor, row_limit in tumor_types.items():
                if tumor == 'normal':
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Negative')]
                else:
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Positive')]
                
                # Use only the specified number of patients for each cohort
                selected_data = filtered_data.iloc[:row_limit][columns_to_average + ['Distance %(3)']].dropna()
                # Convert to numeric and calculate mean
                numeric_means = selected_data.astype(float).mean().to_dict()
                averages_distance.append({'Cohort': tumor.capitalize(), 'Average Distance %(3)': numeric_means.get('Distance %(3)', 0), **numeric_means})
            
            averages_distance_data = pd.DataFrame(averages_distance)
            averages_distance_data.to_csv(base_average_path.format(i), index=False)
            
            # Determine threshold
            sorted_distances = averages_distance_data['Average Distance %(3)'].sort_values(ascending=False)
            second_highest_distance = sorted_distances.iloc[1]
            k = int(np.ceil(second_highest_distance))
            
            # Store k value for later use in offset calculations
            new_data['k_value'] = k
            new_data.to_csv(file_path, index=False)
            
            print(f"Processed and updated {file_path} and {base_average_path.format(i)} successfully.")

    def merge_averages_to_excel(output_dir, num_combinations):
        # Function for natural sorting (e.g., 1,2,3 instead of 1,10,100)
        def natural_sort_key(text):
            return [int(num) if num.isdigit() else num for num in re.split(r'(\d+)', text)]

        # Merge AverageDet CSV files into Average1.xlsx
        # Get a sorted list of all relevant CSV files
        csv_files = sorted(
            [f for f in os.listdir(output_dir) if f.startswith("AverageDet") and f.endswith(".csv")],
            key=natural_sort_key
        )

        # Initialize an empty DataFrame for merging
        merged_df = None

        # Loop through each file and process
        for i, file in enumerate(csv_files, start=1):
            if i > num_combinations:
                break  # Stop after processing the specified number of combinations
            
            # Read the current CSV file
            file_path = os.path.join(output_dir, file)
            df = pd.read_csv(file_path)

            # Select only the "Cohort" and "Average Distance %(3)" columns
            df = df[["Cohort", "Average Distance %(3)"]]

            # Rename the "Average Distance %(3)" column to include the file index
            df.rename(columns={"Average Distance %(3)": f"Average Distance %(3)-{i}"}, inplace=True)

            # Merge with accumulated DataFrame
            if merged_df is None:
                merged_df = df
            else:
                merged_df = pd.merge(merged_df, df, on="Cohort", how="outer")

        # Save the merged DataFrame to an Excel file
        output_file_path = os.path.join(output_dir, "Average1.xlsx")
        merged_df.to_excel(output_file_path, index=False, engine='openpyxl')
        print(f"Merged file saved as: {output_file_path}")

    def process_clinical_data(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range):
        # Step 1: Load the clinical data
        df_1 = pd.read_excel(file_path, sheet_name=sheet_name)
        df_2 = pd.read_excel(file_path, sheet_name=sheet_name)

        # Generate all combinations of 3 biomarkers
        biomarker_combinations = list(combinations(biomarkers, 3))
        if num_combinations > len(biomarker_combinations):
            print(f"Warning: Only {len(biomarker_combinations)} combinations available. Processing {len(biomarker_combinations)} combinations instead of {num_combinations}.")
            num_combinations = len(biomarker_combinations)

        # Directory to save the CSV files
        os.makedirs(output_dir, exist_ok=True)

        # Step 2: Iterate through each combination of biomarkers
        for combo_index, combo in enumerate(biomarker_combinations[:num_combinations], start=1):
            # Select biomarkers for this iteration
            columns_to_select = list(combo)
            print(f"Processing combination {combo_index}: {columns_to_select}")

            # Subset the data
            df_1_subset = df_1[columns_to_select].iloc[:59]  # First 59 rows for normal patients
            df_2_subset = df_2[columns_to_select]           # All rows for cancer patients

            # Initialize a list to store feature vectors
            data_list = []

            # Step 4: Add additional random patient data for analysis
            for index in range(60, random_patient_range + 1):  # User-defined range
                # Select a specific row from df_2
                row_to_add = df_2_subset.iloc[[index]]

                # Combine it with the fixed dataset
                df_combined = pd.concat([df_1_subset] + [row_to_add] * 1, ignore_index=True)

                # Step 5: Log normalize the combined data
                numeric_cols = df_combined.select_dtypes(include=[np.number]).columns
                df_combined[numeric_cols] = np.log(df_combined[numeric_cols])

                # Convert DataFrame to numpy array
                df_subset = df_combined.values

                # Step 6: Calculate Pearson correlation matrix and distance matrix
                corr_matrix = np.corrcoef(df_subset, rowvar=False)
                corr_matrix_df = pd.DataFrame(corr_matrix, columns=columns_to_select, index=columns_to_select)
                distance_matrix = 1 - np.abs(corr_matrix_df)
                distance_matrix_np = distance_matrix.values

                # Step 7: Construct a simplicial complex using Gudhi's RipsComplex
                rips_complex = gudhi.RipsComplex(distance_matrix=distance_matrix_np)
                simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
                persistence = simplex_tree.persistence()

                # Step 8: Extract topological features
                finite_persistence = [(dim, (birth, death)) for dim, (birth, death) in persistence if death != float('inf')]
                one_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 1]
                
                if one_dim_holes:
                    max_life_range_1D = max(death - birth for birth, death in one_dim_holes)
                    avg_birth_1D = np.mean([birth for birth, _ in one_dim_holes])
                    avg_death_1D = np.mean([death for _, death in one_dim_holes])
                    num_one_dim_holes = len(one_dim_holes)
                else:
                    max_life_range_1D = avg_birth_1D = avg_death_1D = num_one_dim_holes = 0

                zero_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 0]
                min_life_range_0D = min(death - birth for birth, death in zero_dim_holes)
                max_life_range_0D = max(death - birth for birth, death in zero_dim_holes)
                avg_death_0D = np.mean([death for _, death in zero_dim_holes])
                num_zero_dim_holes = len(zero_dim_holes) + 1
                range_death_0D = max([death for _, death in zero_dim_holes]) - min([death for _, death in zero_dim_holes])
                median_death_0D = np.median([death for _, death in zero_dim_holes])
                std_death_0D = np.std([death for _, death in zero_dim_holes])

                # Prepare feature vector
                data = {
                    "0-Dim Hole Min Life Range": [min_life_range_0D],
                    "0-Dim Hole Max Life Range": [max_life_range_0D],
                    "Range_Death_0D": [range_death_0D],
                    "Num_0D_Holes": [num_zero_dim_holes],
                    "Median_Death_0D": [median_death_0D],
                    "Std_Death_0D": [std_death_0D],
                    "Avg_Death_0D": [avg_death_0D],
                    "Average Birth (1D)": [avg_birth_1D],
                    "Average Death (1D)": [avg_death_1D],
                    "1-Dim Hole Max Life Range": [max_life_range_1D],
                    "Number of 1-Dimensional Holes": [num_one_dim_holes]
                }

                # Append to list
                df_features = pd.DataFrame(data, index=[f"P_{index}"])
                data_list.append(df_features)

            # Concatenate all feature vectors for the current combination
            final_df = pd.concat(data_list)

            # Add additional clinical information
            additional_columns = ['Patient ID #', 'Cohort', 'AJCC Stage']
            df_clinical = pd.read_excel(file_path, sheet_name=sheet_name)
            df_additional_info = df_clinical[additional_columns].iloc[60:random_patient_range + 1]
            additional_info_index = [f"P_{i}" for i in range(60, random_patient_range + 1)]
            df_additional_info.index = additional_info_index

            # Merge with topological data
            df_combined = pd.concat([df_additional_info, final_df], axis=1)

            # Append biomarkers as the last row
            selected_biomarkers_row = pd.DataFrame(
                {col: None for col in df_combined.columns}, 
                index=["Selected Biomarkers"]
            )
            selected_biomarkers_row.iloc[0, :len(columns_to_select)] = columns_to_select
            df_combined = pd.concat([df_combined, selected_biomarkers_row])

            # Save to a CSV file
            csv_file_path = os.path.join(output_dir, f"FeatureVectorTissueDet{combo_index}.csv")
            df_combined.to_csv(csv_file_path, float_format="%f", index=True)
            print(f"Saved: {csv_file_path}")

        # Step 9: Calculate averages and generate Average1.xlsx
        calculate_averages(output_dir, tumor_types, total_positive_patients, num_combinations)
        merge_averages_to_excel(output_dir, num_combinations)

    # ==================================================
    # Part 2: Formation of Positives1.xlsx and Sensitivity Outcomes
    # ==================================================

    def apply_offset_and_create_positives(output_dir, tumor_types_dict, num_combinations):
        # Create offset directories and apply different thresholds
        for offset_index, offset in enumerate(range(-20, 20), start=1):
            offset_dir = os.path.join(output_dir, f'Positives{offset_index}')
            os.makedirs(offset_dir, exist_ok=True)
            
            print(f"Processing offset {offset} (index {offset_index})")
            
            # Copy the base files to the offset directory
            for i in range(1, num_combinations + 1):
                src_file = os.path.join(output_dir, f'FeatureVectorTissueDet{i}.csv')
                dst_file = os.path.join(offset_dir, f'FeatureVectorTissueDet{i}.csv')
                if os.path.exists(src_file):
                    shutil.copy2(src_file, dst_file)
            
            # Copy Average1.xlsx to the offset directory
            src_avg = os.path.join(output_dir, 'Average1.xlsx')
            dst_avg = os.path.join(offset_dir, 'Average1.xlsx')
            if os.path.exists(src_avg):
                shutil.copy2(src_avg, dst_avg)
            
            # Process each FeatureVector file with the offset threshold
            for i in range(1, num_combinations + 1):
                input_file_path = os.path.join(offset_dir, f'FeatureVectorTissueDet{i}.csv')
                try:
                    # Load the data
                    new_data = pd.read_csv(input_file_path)
                    
                    # Get the k value
                    k = new_data['k_value'].iloc[0]
                    
                    # Apply offset to threshold
                    threshold = k + offset
                    
                    # Localize cancer tissue with offset threshold
                    new_data['Result(Tissue Detection)'] = ['Positive' if value > threshold else 'Negative' for value in new_data['Distance %(3)']]
                    new_data['Cancer tissue'] = new_data.apply(
                        lambda row: 'Positive' if row['Result(Cancer)'] == 'Positive' and row['Result(Tissue Detection)'] == 'Positive' else 'Negative', 
                        axis=1
                    )
                    
                    # Save the updated file
                    new_data.to_csv(input_file_path, index=False)
                    
                except FileNotFoundError:
                    print(f"File not found: {input_file_path}, skipping...")
            
            # Now create positives count files for this offset
            merge_positives_to_excel(offset_dir, tumor_types_dict, num_combinations, offset_index)
            calculate_metrics(offset_dir, tumor_types_dict, total_positive_patients, num_combinations, offset_index)
            get_top_10_metrics(offset_dir, tumor_types_dict, num_combinations, offset_index)

    def merge_positives_to_excel(output_dir, tumor_types_dict, num_combinations, offset_index=None):
        # Directory paths
        input_dir = output_dir
        positives_dir = os.path.join(input_dir, 'Positives')

        # Ensure the output directory exists
        os.makedirs(positives_dir, exist_ok=True)

        # Loop through files 1 to N
        for i in range(1, num_combinations + 1):
            input_file_path = os.path.join(input_dir, f'FeatureVectorTissueDet{i}.csv')
            output_file_path = os.path.join(positives_dir, f'positives_countDet{i}.csv')

            try:
                # Step 1: Load the CSV file
                new_data = pd.read_csv(input_file_path, header=0)

                # Step 2: Initialize an empty list to store results
                results = []

                # Step 3: Count the number of positives for each Cohort and each AJCC Stage
                # USE ALL PATIENTS FOR EACH COHORT (NOT LIMITED BY tumor_types_dict)
                for tumor_type in tumor_types_dict.keys():
                    if tumor_type == 'normal':
                        continue  # Skip normal cohort for positive counts
                    
                    # Filter data for the current Cohort - USE ALL PATIENTS
                    tumor_data = new_data[new_data['Cohort'].str.lower() == tumor_type.lower()]

                    # Count positives for each AJCC stage (I, II, III)
                    for stage in ['I', 'II', 'III']:
                        stage_data = tumor_data[tumor_data['AJCC Stage'] == stage]

                        # Count positives for 'Cancer tissue' column
                        positive_count = stage_data[stage_data['Cancer tissue'] == 'Positive'].shape[0]

                        # Append result to the list
                        results.append({'Cohort': tumor_type.capitalize(), 'AJCC Stage': stage, 'Positive Count': positive_count})

                    # Count positives for all AJCC stages combined
                    total_positive_count = tumor_data[tumor_data['Cancer tissue'] == 'Positive'].shape[0]
                    results.append({'Cohort': tumor_type.capitalize(), 'AJCC Stage': 'All Stages', 'Positive Count': total_positive_count})

                # Step 4: Convert the results into a DataFrame and save to CSV
                results_df = pd.DataFrame(results)
                results_df.to_csv(output_file_path, index=False)

                print(f"Processed: {input_file_path} → Saved: {output_file_path}")

            except FileNotFoundError:
                print(f"File not found: {input_file_path}, skipping...")

        # Merge all positives_countDet CSV files into Positives1.xlsx
        # Function for natural sorting (e.g., 1,2,3 instead of 1,10,100)
        def natural_sort_key(text):
            return [int(num) if num.isdigit() else num for num in re.split(r'(\d+)', text)]

        # Get a sorted list of all relevant CSV files
        csv_files = sorted(
            [f for f in os.listdir(positives_dir) if f.startswith("positives_countDet") and f.endswith(".csv")],
            key=natural_sort_key
        )

        # Initialize an empty list to store DataFrames
        dfs = []

        # Loop through each file and process it
        for i, file in enumerate(csv_files, start=1):
            if i > num_combinations:
                break  # Stop after processing the specified number of combinations
            
            # Read the current CSV file
            file_path = os.path.join(positives_dir, file)
            df = pd.read_csv(file_path)

            # Filter rows based on conditions - ONLY INCLUDE THE SPECIFIED COHORTS
            df = df[(df["AJCC Stage"] == "All Stages") & (df["Cohort"].isin([c.capitalize() for c in tumor_types_dict.keys() if c != 'normal']))]

            # Rename the "Positive Count" column to include the file index
            df.rename(columns={"Positive Count": f"Positive Det {i}"}, inplace=True)

            # Append the processed DataFrame to the list
            dfs.append(df)

        # Merge all DataFrames on "Cohort" and "AJCC Stage"
        if dfs:
            merged_df = dfs[0]
            for df in dfs[1:]:
                merged_df = pd.merge(merged_df, df, on=["Cohort", "AJCC Stage"], how="outer")

            # Sort the final DataFrame by "Cohort" and "AJCC Stage" for consistency
            merged_df = merged_df.sort_values(by=["Cohort", "AJCC Stage"]).reset_index(drop=True)

            # Add "Total Positive Patients" column - USE THE total_positive_patients DICTIONARY
            merged_df["Total Positive Patients"] = merged_df["Cohort"].map(total_positive_patients)

            # Reorder columns to place "Total Positive Patients" after "AJCC Stage"
            cols = ["Cohort", "AJCC Stage", "Total Positive Patients"] + [col for col in merged_df.columns if col.startswith("Positive Det")]
            merged_df = merged_df[cols]

            # Save the merged DataFrame to an Excel file
            if offset_index:
                output_file_path = os.path.join(positives_dir, f"Positives{offset_index}.xlsx")
            else:
                output_file_path = os.path.join(positives_dir, "Positives1.xlsx")
            merged_df.to_excel(output_file_path, index=False, engine='openpyxl')

            print(f"Merged file saved as: {output_file_path}")
        else:
            print("No data to merge for Positives1.xlsx")

    def calculate_metrics(output_dir, tumor_types_dict, total_positive_patients, num_combinations, offset_index=None):
        # Determine the positives file path based on offset index
        if offset_index:
            positives_file_path = os.path.join(output_dir, 'Positives', f'Positives{offset_index}.xlsx')
        else:
            positives_file_path = os.path.join(output_dir, 'Positives', 'Positives1.xlsx')
            
        averages_file_path = os.path.join(output_dir, 'Average1.xlsx')

        positives_df = pd.read_excel(positives_file_path)
        averages_df = pd.read_excel(averages_file_path)

        # Get the list of Positive Count columns
        positive_count_columns = [col for col in positives_df.columns if col.startswith("Positive Det")]

        # Initialize columns for results
        results = {"Sensitivity": [], "Precision": [], "F1_Score": [], "Accuracy": [], "Cohort": []}

        # Total number of positive patients across all Cohorts - USE THE total_positive_patients DICTIONARY
        total_patients = positives_df["Total Positive Patients"].sum()

        # Loop through each Positive Count column
        for i, column in enumerate(positive_count_columns, start=1):
            if i > num_combinations:
                break  # Stop after processing the specified number of combinations
            
            # Determine the Cohort type based on the highest value in the corresponding "Average Distance %(3)-i" column
            avg_column = f"Average Distance %(3)-{i}"
            
            if avg_column not in averages_df.columns:
                print(f"Column {avg_column} not found in Averages file, skipping...")
                continue  # Skip this iteration if column is missing
            
            max_value_row = averages_df[avg_column].idxmax()
            Cohort_type = averages_df.loc[max_value_row, "Cohort"]

            # Ensure the Cohort exists in positives_df
            Cohort_row = positives_df[positives_df["Cohort"] == Cohort_type]
            if Cohort_row.empty:
                print(f"Cohort {Cohort_type} not found in Positives file, skipping...")
                continue
            
            Cohort_row = Cohort_row.iloc[0]  # Convert to Series
            
            # Extract necessary values
            TP = Cohort_row[column]  # True Positives
            FN = Cohort_row["Total Positive Patients"] - TP  # False Negatives
            
            # Calculate FP and TN using the total_positive_patients dictionary
            # For FP: Sum of all positives detected minus TP for this cohort
            FP = positives_df[column].sum() - TP
            
            # For TN: Total patients minus (TP + FP + FN)
            # Total patients is the sum of all values in total_positive_patients
            TN = total_patients - (TP + FP + FN)

            # Calculate Sensitivity, Precision, F1 Score, and Accuracy
            sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            f1_score = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
            accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
            
            # Append results
            results["Sensitivity"].append(sensitivity)
            results["Precision"].append(precision)
            results["F1_Score"].append(f1_score)
            results["Accuracy"].append(accuracy)
            results["Cohort"].append(Cohort_type)

            # Write results into the DataFrame
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Sensitivity {i}"] = sensitivity
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Precision {i}"] = precision
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"F1 Score {i}"] = f1_score
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Accuracy {i}"] = accuracy

        # Save the results back to the Excel file
        positives_df.to_excel(positives_file_path, index=False, engine='openpyxl')

        print(f"Results saved to: {positives_file_path}")

    def get_top_10_metrics(output_dir, tumor_types_dict, num_combinations, offset_index=None):
        if offset_index:
            excel_file_path = os.path.join(output_dir, 'Positives', f'Positives{offset_index}.xlsx')
        else:
            excel_file_path = os.path.join(output_dir, 'Positives', 'Positives1.xlsx')
            
        csv_file_folder = output_dir
        output_path = os.path.join(output_dir, 'Positives')

        # Read the Excel file
        excel_df = pd.read_excel(excel_file_path)

        # Function for natural sorting (e.g., 1,2,3 instead of 1,10,100)
        def natural_sort_key(text):
            return [int(num) if num.isdigit() else num for num in re.split(r'(\d+)', text)]

        # Function to get selected biomarkers from the appropriate CSV file
        def get_selected_biomarkers(csv_file_name, col_start=1, col_end=4):
            csv_file_path = os.path.join(csv_file_folder, csv_file_name)
            if os.path.exists(csv_file_path):
                csv_df = pd.read_csv(csv_file_path)
                return csv_df.iloc[-1, col_start:col_end].values.tolist()  # Convert NumPy array to Python list
            else:
                print(f"Warning: File {csv_file_name} not found.")
                return ["NA"]  # Return "NA" instead of an empty list

        # Initialize a list to store results
        results = []

        # Process each Cohort (row) in the Excel file
        for idx, row in excel_df.iterrows():
            Cohort = row['Cohort']
            print(f"Processing Cohort: {Cohort}")
            
            # Initialize the list for this Cohort
            Cohort_results = {
                'Cohort': Cohort,
                'Offset': offset_index - 20 if offset_index else 0  # Convert to actual offset value (-20 to +20)
            }
            
            # Add columns for top 10 results
            for i in range(1, 11):
                Cohort_results[f'Top {i} F1 Score'] = 'NA'
                Cohort_results[f'Top {i} Accuracy'] = 'NA'
                Cohort_results[f'Top {i} Precision'] = 'NA'
                Cohort_results[f'Top {i} Sensitivity'] = 'NA'
                Cohort_results[f'Top {i} Biomarkers'] = 'NA'
            
            # Generate ordered metric column groups
            metric_groups = []
            for i in range(1, num_combinations + 1):
                metric_groups.append({
                    'f1_score': f'F1 Score {i}',
                    'accuracy': f'Accuracy {i}',
                    'precision': f'Precision {i}',
                    'sensitivity': f'Sensitivity {i}'
                })
            
            # Store valid metric groups
            valid_metrics = []
            
            for group in metric_groups:
                # Check if all columns exist in the row
                if not all(col in row for col in group.values()):
                    continue
                
                f1_score_value = row[group['f1_score']]
                accuracy_value = row[group['accuracy']]
                precision_value = row[group['precision']]
                sensitivity_value = row[group['sensitivity']]
                
                if pd.notna(f1_score_value) and pd.notna(accuracy_value) and f1_score_value > 0 and accuracy_value > 0:
                    # Determine which CSV file to read based on the metric number
                    metric_num = int(group['f1_score'].split(' ')[2])
                    csv_file_name = f'FeatureVectorTissueDet{metric_num}.csv'
                    
                    # Get selected biomarkers for the current metric group
                    biomarkers = get_selected_biomarkers(csv_file_name)
                    
                    # Store the result
                    valid_metrics.append({
                        'F1_Score': f1_score_value,
                        'Accuracy': accuracy_value,
                        'Precision': precision_value,
                        'Sensitivity': sensitivity_value,
                        'Biomarkers': biomarkers,
                        'Metric_Number': metric_num
                    })
            
            # If valid metrics exist, get the top 10 sorted by highest F1 Score (primary) and Accuracy (secondary)
            if valid_metrics:
                sorted_metrics = sorted(valid_metrics, key=lambda x: (x['F1_Score'], x['Accuracy']), reverse=True)[:10]
                
                # Update the Cohort result with the top 10 metrics
                for i, res in enumerate(sorted_metrics):
                    Cohort_results[f'Top {i+1} F1 Score'] = res['F1_Score']
                    Cohort_results[f'Top {i+1} Accuracy'] = res['Accuracy']
                    Cohort_results[f'Top {i+1} Precision'] = res['Precision']
                    Cohort_results[f'Top {i+1} Sensitivity'] = res['Sensitivity']
                    Cohort_results[f'Top {i+1} Biomarkers'] = res['Biomarkers']
            
            # Append the result for the current Cohort
            results.append(Cohort_results)

        # Create a DataFrame for the results
        results_df = pd.DataFrame(results)

        # Ensure output path exists
        os.makedirs(output_path, exist_ok=True)

        # Define output file path
        if offset_index:
            output_file_path = os.path.join(output_path, f'Top_10_Metrics_{offset_index}.xlsx')
        else:
            output_file_path = os.path.join(output_path, 'Top_10_Metrics.xlsx')

        # Save the results to Excel
        results_df.to_excel(output_file_path, index=False, engine='openpyxl')

        print(f"Results saved to {output_file_path}")

    # ==================================================
    # Part 3: Find Top 10 F1 Scores Across All Offsets
    # ==================================================

    def find_top_10_f1_scores_across_offsets(output_dir, tumor_types_dict):
        """
        Find the top 10 F1 scores across all offset files and provide the respective
        biomarker combinations and offset information for each cohort.
        """
        all_results = []
        
        # Get all Top_10_Metrics files
        metrics_files = []
        for offset_index in range(1, 41):  # Offsets from -20 to +19 (40 files)
            file_path = os.path.join(output_dir, f'Positives{offset_index}', 'Positives', f'Top_10_Metrics_{offset_index}.xlsx')
            if os.path.exists(file_path):
                actual_offset = offset_index - 21  # Convert to actual offset value (-20 to +19)
                metrics_files.append((actual_offset, file_path))
        
        # Process each cohort
        for cohort in tumor_types_dict.keys():
            if cohort == 'normal':
                continue
                
            cohort_results = []
            
            for offset, file_path in metrics_files:
                try:
                    df = pd.read_excel(file_path)
                    cohort_data = df[df['Cohort'] == cohort.capitalize()]
                    
                    if not cohort_data.empty:
                        # Get all top 10 results for this cohort and offset
                        for i in range(1, 10):
                            f1_score = cohort_data[f'Top {i} F1 Score'].values[0]
                            if pd.notna(f1_score) and f1_score != 'NA':
                                cohort_results.append({
                                    'Cohort': cohort.capitalize(),
                                    'F1_Score': f1_score,
                                    'Accuracy': cohort_data[f'Top {i} Accuracy'].values[0],
                                    'Precision': cohort_data[f'Top {i} Precision'].values[0],
                                    'Sensitivity': cohort_data[f'Top {i} Sensitivity'].values[0],
                                    'Biomarkers': cohort_data[f'Top {i} Biomarkers'].values[0],
                                    'Offset': offset,
                                    'Rank': i
                                })
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")
                    continue
            
            # Sort all results for this cohort by F1 score (descending)
            if cohort_results:
                cohort_results_sorted = sorted(cohort_results, key=lambda x: x['F1_Score'], reverse=True)[:10]
                all_results.extend(cohort_results_sorted)
        
        # Create DataFrame and save to Excel
        if all_results:
            results_df = pd.DataFrame(all_results)
            output_file = os.path.join(output_dir, 'Top_10_F1_ScoresBased_LocPerformance_ALL.xlsx')
            results_df.to_excel(output_file, index=False, engine='openpyxl')
            print(f"Top 10 F1 scores across offsets saved to: {output_file}")
            
            # Print results
            print("\n=== TOP 10 F1 SCORES ACROSS ALL OFFSETS AND TISSUE LOCALIZATION PERFORMANCE ===")
            for cohort in tumor_types_dict.keys():
                if cohort == 'normal':
                    continue
                    
                cohort_data = results_df[results_df['Cohort'] == cohort.capitalize()]
                if not cohort_data.empty:
                    print(f"\nCohort: {cohort.capitalize()}")
                    print("-" * 80)
                    for idx, row in cohort_data.iterrows():
                        print(f"Rank: {row['Rank']}")
                        print(f"F1 Score: {row['F1_Score']:.6f}")
                        print(f"Accuracy: {row['Accuracy']:.6f}")
                        print(f"Precision: {row['Precision']:.6f}")
                        print(f"Sensitivity: {row['Sensitivity']:.6f}")
                        print(f"Biomarkers: {row['Biomarkers']}")
                        print(f"Offset: {row['Offset']}")
                        print("-" * 40)
        else:
            print("No F1 scores found across offsets.")
    
    # ==================================================

    # Main Execution
    # ==================================================

    # Part 1: Preliminary Analysis and Formation of Average1.xlsx (only once)
    process_clinical_data(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range)

    # Part 2: Formation of Positives1.xlsx and Metrics Outcomes with offset thresholds
    apply_offset_and_create_positives(output_dir, tumor_types, num_combinations)
    
    # Part 3: Find top 10 F1 scores across all offsets
    find_top_10_f1_scores_across_offsets(output_dir, tumor_types)

# Example usage (Provide the required user-defined values)
file_path = r'D:\Cancer Detection folder\Test 3\Clinical cancer validation data.xlsx'
sheet_name = 'Normal and Cancer'
biomarkers = ['sc-kit/SCFR', 'CA-125', 'CA 19-9', 'HE4', 'Prolactin', 'SCF', 'Total PSA', 'IGFBP3']
output_dir = r"D:\Cancer Detection folder\Test 3"

tumor_types = {
    'normal': 47,
    'bladder': 9,
    'ovarian': 11,
    'pancreatic': 17
}
total_positive_patients = {
    "Bladder": 26,      
    "Ovarian": 32,      
    "Pancreatic": 39    
}
cohorts_to_analyze = ['Bladder', 'Ovarian', 'Pancreatic']
num_combinations = 56  
random_patient_range = 322 

# Execute the TDA_cancerDetect function
TDAcancer_detect_2(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range, cohorts_to_analyze)

Processing combination 1: ['sc-kit/SCFR', 'CA-125', 'CA 19-9']


C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1235678023.py:259: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, selected_biomarkers_row])


Saved: D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet1.csv
Processing combination 2: ['sc-kit/SCFR', 'CA-125', 'HE4']


C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1235678023.py:259: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, selected_biomarkers_row])


Saved: D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet2.csv
Processing combination 3: ['sc-kit/SCFR', 'CA-125', 'Prolactin']


C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1235678023.py:259: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, selected_biomarkers_row])


... (183 outputs left out)

C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1235678023.py:509: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positives_df.loc[positives_df["Cohort"] == Cohort_type, f"F1 Score {i}"] = f1_score
C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1235678023.py:510: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Accuracy {i}"] = accuracy
C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1235678023.py:507: PerformanceWarning: DataFrame is high

Results saved to: D:\Cancer Detection folder\Test 3\Positives39\Positives\Positives39.xlsx
Processing Cohort: Bladder
Processing Cohort: Ovarian
Processing Cohort: Pancreatic
Results saved to D:\Cancer Detection folder\Test 3\Positives39\Positives\Top_10_Metrics_39.xlsx
Processing offset 19 (index 40)
Processed: D:\Cancer Detection folder\Test 3\Positives40\FeatureVectorTissueDet1.csv → Saved: D:\Cancer Detection folder\Test 3\Positives40\Positives\positives_countDet1.csv
Processed: D:\Cancer Detection folder\Test 3\Positives40\FeatureVectorTissueDet2.csv → Saved: D:\Cancer Detection folder\Test 3\Positives40\Positives\positives_countDet2.csv
Processed: D:\Cancer Detection folder\Test 3\Positives40\FeatureVectorTissueDet3.csv → Saved: D:\Cancer Detection folder\Test 3\Positives40\Positives\positives_countDet3.csv
Processed: D:\Cancer Detection folder\Test 3\Positives40\FeatureVectorTissueDet4.csv → Saved: D:\Cancer Detection folder\Test 3\Positives40\Positives\positives_countDet4.csv
P

C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1235678023.py:509: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positives_df.loc[positives_df["Cohort"] == Cohort_type, f"F1 Score {i}"] = f1_score
C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1235678023.py:510: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Accuracy {i}"] = accuracy
C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1235678023.py:507: PerformanceWarning: DataFrame is high

Results saved to: D:\Cancer Detection folder\Test 3\Positives40\Positives\Positives40.xlsx
Processing Cohort: Bladder
Processing Cohort: Ovarian
Processing Cohort: Pancreatic
Results saved to D:\Cancer Detection folder\Test 3\Positives40\Positives\Top_10_Metrics_40.xlsx
Top 10 F1 scores across offsets saved to: D:\Cancer Detection folder\Test 3\Top_10_F1_ScoresBased_LocPerformance_ALL.xlsx

=== TOP 10 F1 SCORES ACROSS ALL OFFSETS AND TISSUE LOCALIZATION PERFORMANCE ===

Cohort: Bladder
--------------------------------------------------------------------------------
Rank: 1
F1 Score: 0.608696
Accuracy: 0.721649
Precision: 0.488372
Sensitivity: 0.807692
Biomarkers: ['sc-kit/SCFR', 'HE4', 'Total PSA']
Offset: -1
----------------------------------------
Rank: 1
F1 Score: 0.608696
Accuracy: 0.814433
Precision: 0.700000
Sensitivity: 0.538462
Biomarkers: ['sc-kit/SCFR', 'HE4', 'Total PSA']
Offset: 0
----------------------------------------
Rank: 2
F1 Score: 0.607143
Accuracy: 0.773196
Precisi

In [7]:
# Tissue of Origin Localization Module for a Specific Cohort and Biomarker Combination
# Run the function/module TDAcancer_detect_2_SpecificCohort to evaluate and validate tissue localization 
# for a specific cohort and biomarker combination across the entire dataset (local calibration + test). 
# The performance outcomes obtained here validate the results from the local calibration samples 
# for the corresponding cancer type, biomarker combination, and threshold offset.

import os
import re
import pandas as pd
import numpy as np
import gudhi
from itertools import combinations

def TDAcancer_detect_2_specificCohort(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range, cohorts_to_analyze, offset=-1, specific_cohort=None, specific_biomarkers=None):
    
    # ==================================================
    # Part 1: Preliminary Analysis and Formation of Average1.xlsx
    # ==================================================

    def calculate_averages(output_dir, tumor_types, total_positive_patients, num_combinations, offset):
        # File paths
        cancer_result_file = os.path.join(output_dir, 'Clinical cancer data_features.csv')
        base_feature_vector_path = os.path.join(output_dir, 'FeatureVectorTissueDet{}.csv')
        base_average_path = os.path.join(output_dir, 'AverageDet{}.csv')

        # Load the cancer results file
        cancer_data = pd.read_csv(cancer_result_file)

        # Columns for averaging
        columns_to_average = ['0-Dim Hole Min Life Range', 'Range_Death_0D']

        # Process files from FeatureVectorTissueDet1.csv to FeatureVectorTissueDetN.csv
        for i in range(1, num_combinations + 1):
            file_path = base_feature_vector_path.format(i)
            try:
                new_data = pd.read_csv(file_path)
            except FileNotFoundError:
                print(f"File {file_path} not found, skipping.")
                continue
            
            # Add 'Result(Cancer)' column
            new_data['Result(Cancer)'] = cancer_data['Result(Cancer)']
            new_data.to_csv(file_path, index=False)
            
            # Compute cohort averages
            averages_list = []
            for tumor, row_limit in tumor_types.items():
                if tumor == 'normal':
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Negative')]
                else:
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Positive')]
                
                selected_data = filtered_data.iloc[:row_limit][columns_to_average].dropna()
                averages_list.append({'Cohort': tumor.capitalize(), **selected_data.mean().to_dict()})
            
            averages_data = pd.DataFrame(averages_list)
            normal_averages = averages_data[averages_data['Cohort'].str.lower() == 'normal']
            reference_point = (
                float(normal_averages['0-Dim Hole Min Life Range'].values[0]),
                float(normal_averages['Range_Death_0D'].values[0])
            )
            
            # Compute Euclidean distance
            X = new_data[['0-Dim Hole Min Life Range']].values
            y = new_data['Range_Death_0D'].values
            distances = [np.sqrt((X[i][0] - reference_point[0]) ** 2 + (y[i] - reference_point[1]) ** 2) for i in range(len(X))]
            new_data['Distance %(3)'] = [d * 100 for d in distances]
            new_data.to_csv(file_path, index=False)
            
            # Compute distance averages
            averages_distance = []
            for tumor, row_limit in tumor_types.items():
                if tumor == 'normal':
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Negative')]
                else:
                    filtered_data = new_data[(new_data['Cohort'].str.strip().str.lower() == tumor) & (new_data['Result(Cancer)'] == 'Positive')]
                
                selected_data = filtered_data.iloc[:row_limit][columns_to_average + ['Distance %(3)']].dropna()
                averages_distance.append({'Cohort': tumor.capitalize(), 'Average Distance %(3)': selected_data['Distance %(3)'].mean(), **selected_data.mean().to_dict()})
            
            averages_distance_data = pd.DataFrame(averages_distance)
            averages_distance_data.to_csv(base_average_path.format(i), index=False)
            
            # Determine threshold
            sorted_distances = averages_distance_data['Average Distance %(3)'].sort_values(ascending=False)
            second_highest_distance = sorted_distances.iloc[1]
            k = int(np.ceil(second_highest_distance))
            
            # Localize cancer tissue with user-defined offset
            new_data['Result(Tissue Detection)'] = ['Positive' if value > k + offset else 'Negative' for value in new_data['Distance %(3)']]
            new_data['Cancer tissue'] = new_data.apply(lambda row: 'Positive' if row['Result(Cancer)'] == 'Positive' and row['Result(Tissue Detection)'] == 'Positive' else 'Negative', axis=1)
            new_data.to_csv(file_path, index=False)
            
            print(f"Processed and updated {file_path} and {base_average_path.format(i)} successfully.")

    def merge_averages_to_excel(output_dir, num_combinations):
        # Function for natural sorting (e.g., 1,2,3 instead of 1,10,100)
        def natural_sort_key(text):
            return [int(num) if num.isdigit() else num for num in re.split(r'(\d+)', text)]

        # Merge AverageDet CSV files into Average1.xlsx
        # Get a sorted list of all relevant CSV files
        csv_files = sorted(
            [f for f in os.listdir(output_dir) if f.startswith("AverageDet") and f.endswith(".csv")],
            key=natural_sort_key
        )

        # Initialize an empty DataFrame for merging
        merged_df = None

        # Loop through each file and process
        for i, file in enumerate(csv_files, start=1):
            if i > num_combinations:
                break  # Stop after processing the specified number of combinations
            
            # Read the current CSV file
            file_path = os.path.join(output_dir, file)
            df = pd.read_csv(file_path)

            # Select only the "Cohort" and "Average Distance %(3)" columns
            df = df[["Cohort", "Average Distance %(3)"]]

            # Rename the "Average Distance %(3)" column to include the file index
            df.rename(columns={"Average Distance %(3)": f"Average Distance %(3)-{i}"}, inplace=True)

            # Merge with accumulated DataFrame
            if merged_df is None:
                merged_df = df
            else:
                merged_df = pd.merge(merged_df, df, on="Cohort", how="outer")

        # Save the merged DataFrame to an Excel file
        output_file_path = os.path.join(output_dir, "Average1.xlsx")
        merged_df.to_excel(output_file_path, index=False, engine='openpyxl')
        print(f"Merged file saved as: {output_file_path}")

    def process_clinical_data(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range, offset):
        # Step 1: Load the clinical data
        df_1 = pd.read_excel(file_path, sheet_name=sheet_name)
        df_2 = pd.read_excel(file_path, sheet_name=sheet_name)

        # Generate all combinations of 3 biomarkers
        biomarker_combinations = list(combinations(biomarkers, 3))
        if num_combinations > len(biomarker_combinations):
            print(f"Warning: Only {len(biomarker_combinations)} combinations available. Processing {len(biomarker_combinations)} combinations instead of {num_combinations}.")
            num_combinations = len(biomarker_combinations)

        # Directory to save the CSV files
        os.makedirs(output_dir, exist_ok=True)

        # Step 2: Iterate through each combination of biomarkers
        for combo_index, combo in enumerate(biomarker_combinations[:num_combinations], start=1):
            # Select biomarkers for this iteration
            columns_to_select = list(combo)
            print(f"Processing combination {combo_index}: {columns_to_select}")

            # Subset the data
            df_1_subset = df_1[columns_to_select].iloc[:59]  # First 59 rows for normal patients
            df_2_subset = df_2[columns_to_select]           # All rows for cancer patients

            # Initialize a list to store feature vectors
            data_list = []

            # Step 4: Add additional random patient data for analysis
            for index in range(60, random_patient_range + 1):  # User-defined range
                # Select a specific row from df_2
                row_to_add = df_2_subset.iloc[[index]]

                # Combine it with the fixed dataset
                df_combined = pd.concat([df_1_subset] + [row_to_add] * 1, ignore_index=True)

                # Step 5: Log normalize the combined data
                numeric_cols = df_combined.select_dtypes(include=[np.number]).columns
                df_combined[numeric_cols] = np.log(df_combined[numeric_cols])

                # Convert DataFrame to numpy array
                df_subset = df_combined.values

                # Step 6: Calculate Pearson correlation matrix and distance matrix
                corr_matrix = np.corrcoef(df_subset, rowvar=False)
                corr_matrix_df = pd.DataFrame(corr_matrix, columns=columns_to_select, index=columns_to_select)
                distance_matrix = 1 - np.abs(corr_matrix_df)
                distance_matrix_np = distance_matrix.values

                # Step 7: Construct a simplicial complex using Gudhi's RipsComplex
                rips_complex = gudhi.RipsComplex(distance_matrix=distance_matrix_np)
                simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
                persistence = simplex_tree.persistence()

                # Step 8: Extract topological features
                finite_persistence = [(dim, (birth, death)) for dim, (birth, death) in persistence if death != float('inf')]
                one_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 1]
                
                if one_dim_holes:
                    max_life_range_1D = max(death - birth for birth, death in one_dim_holes)
                    avg_birth_1D = np.mean([birth for birth, _ in one_dim_holes])
                    avg_death_1D = np.mean([death for _, death in one_dim_holes])
                    num_one_dim_holes = len(one_dim_holes)
                else:
                    max_life_range_1D = avg_birth_1D = avg_death_1D = num_one_dim_holes = 0

                zero_dim_holes = [(birth, death) for dim, (birth, death) in finite_persistence if dim == 0]
                min_life_range_0D = min(death - birth for birth, death in zero_dim_holes)
                max_life_range_0D = max(death - birth for birth, death in zero_dim_holes)
                avg_death_0D = np.mean([death for _, death in zero_dim_holes])
                num_zero_dim_holes = len(zero_dim_holes) + 1
                range_death_0D = max([death for _, death in zero_dim_holes]) - min([death for _, death in zero_dim_holes])
                median_death_0D = np.median([death for _, death in zero_dim_holes])
                std_death_0D = np.std([death for _, death in zero_dim_holes])

                # Prepare feature vector
                data = {
                    "0-Dim Hole Min Life Range": [min_life_range_0D],
                    "0-Dim Hole Max Life Range": [max_life_range_0D],
                    "Range_Death_0D": [range_death_0D],
                    "Num_0D_Holes": [num_zero_dim_holes],
                    "Median_Death_0D": [median_death_0D],
                    "Std_Death_0D": [std_death_0D],
                    "Avg_Death_0D": [avg_death_0D],
                    "Average Birth (1D)": [avg_birth_1D],
                    "Average Death (1D)": [avg_death_1D],
                    "1-Dim Hole Max Life Range": [max_life_range_1D],
                    "Number of 1-Dimensional Holes": [num_one_dim_holes]
                }

                # Append to list
                df_features = pd.DataFrame(data, index=[f"P_{index}"])
                data_list.append(df_features)

            # Concatenate all feature vectors for the current combination
            final_df = pd.concat(data_list)

            # Add additional clinical information
            additional_columns = ['Patient ID #', 'Cohort', 'AJCC Stage']
            df_clinical = pd.read_excel(file_path, sheet_name=sheet_name)
            df_additional_info = df_clinical[additional_columns].iloc[60:random_patient_range + 1]
            additional_info_index = [f"P_{i}" for i in range(60, random_patient_range + 1)]
            df_additional_info.index = additional_info_index

            # Merge with topological data
            df_combined = pd.concat([df_additional_info, final_df], axis=1)

            # Append biomarkers as the last row
            selected_biomarkers_row = pd.DataFrame(
                {col: None for col in df_combined.columns}, 
                index=["Selected Biomarkers"]
            )
            selected_biomarkers_row.iloc[0, :len(columns_to_select)] = columns_to_select
            df_combined = pd.concat([df_combined, selected_biomarkers_row])

            # Save to a CSV file
            csv_file_path = os.path.join(output_dir, f"FeatureVectorTissueDet{combo_index}.csv")
            df_combined.to_csv(csv_file_path, float_format="%f", index=True)
            print(f"Saved: {csv_file_path}")

        # Step 9: Calculate averages and generate Average1.xlsx
        calculate_averages(output_dir, tumor_types, total_positive_patients, num_combinations, offset)
        merge_averages_to_excel(output_dir, num_combinations)

    # ==================================================
    # Part 2: Formation of Positives1.xlsx and Performance Metrics
    # ==================================================

    def merge_positives_to_excel(output_dir, tumor_types, num_combinations):
        # Directory paths
        input_dir = output_dir
        positives_dir = os.path.join(input_dir, 'Positives')

        # Ensure the output directory exists
        os.makedirs(positives_dir, exist_ok=True)

        # Loop through files 1 to N
        for i in range(1, num_combinations + 1):
            input_file_path = os.path.join(input_dir, f'FeatureVectorTissueDet{i}.csv')
            output_file_path = os.path.join(positives_dir, f'positives_countDet{i}.csv')

            try:
                # Step 1: Load the CSV file
                new_data = pd.read_csv(input_file_path, header=0)

                # Step 2: Initialize an empty list to store results
                results = []

                # Step 3: Count the number of positives for each Cohort and each AJCC Stage
                for tumor_type in tumor_types:
                    # Filter data for the current Cohort
                    tumor_data = new_data[new_data['Cohort'] == tumor_type]

                    # Count positives for each AJCC stage (I, II, III)
                    for stage in ['I', 'II', 'III']:
                        stage_data = tumor_data[tumor_data['AJCC Stage'] == stage]

                        # Count positives for 'Cancer tissue' column
                        positive_count = stage_data[stage_data['Cancer tissue'] == 'Positive'].shape[0]

                        # Append result to the list
                        results.append({'Cohort': tumor_type, 'AJCC Stage': stage, 'Positive Count': positive_count})

                    # Count positives for all AJCC stages combined
                    total_positive_count = tumor_data[tumor_data['Cancer tissue'] == 'Positive'].shape[0]
                    results.append({'Cohort': tumor_type, 'AJCC Stage': 'All Stages', 'Positive Count': total_positive_count})

                # Step 4: Convert the results into a DataFrame and save to CSV
                results_df = pd.DataFrame(results)
                results_df.to_csv(output_file_path, index=False)

                print(f"Processed: {input_file_path} → Saved: {output_file_path}")

            except FileNotFoundError:
                print(f"File not found: {input_file_path}, skipping...")

        # Merge all positives_countDet CSV files into Positives1.xlsx
        # Function for natural sorting (e.g., 1,2,3 instead of 1,10,100)
        def natural_sort_key(text):
            return [int(num) if num.isdigit() else num for num in re.split(r'(\d+)', text)]

        # Get a sorted list of all relevant CSV files
        csv_files = sorted(
            [f for f in os.listdir(positives_dir) if f.startswith("positives_countDet") and f.endswith(".csv")],
            key=natural_sort_key
        )

        # Initialize an empty list to store DataFrames
        dfs = []

        # Loop through each file and process it
        for i, file in enumerate(csv_files, start=1):
            if i > num_combinations:
                break  # Stop after processing the specified number of combinations
            
            # Read the current CSV file
            file_path = os.path.join(positives_dir, file)
            df = pd.read_csv(file_path)

            # Filter rows based on conditions
            df = df[(df["AJCC Stage"] == "All Stages") & (df["Cohort"].isin(tumor_types))]

            # Rename the "Positive Count" column to include the file index
            df.rename(columns={"Positive Count": f"Positive Det {i}"}, inplace=True)

            # Append the processed DataFrame to the list
            dfs.append(df)

        # Merge all DataFrames on "Cohort" and "AJCC Stage"
        merged_df = dfs[0]
        for df in dfs[1:]:
            merged_df = pd.merge(merged_df, df, on=["Cohort", "AJCC Stage"], how="outer")

        # Sort the final DataFrame by "Cohort" and "AJCC Stage" for consistency
        merged_df = merged_df.sort_values(by=["Cohort", "AJCC Stage"]).reset_index(drop=True)

        # Add "Total Positive Patients" column
        merged_df["Total Positive Patients"] = merged_df["Cohort"].map(total_positive_patients)

        # Reorder columns to place "Total Positive Patients" after "AJCC Stage"
        cols = ["Cohort", "AJCC Stage", "Total Positive Patients"] + [col for col in merged_df.columns if col.startswith("Positive Det")]
        merged_df = merged_df[cols]

        # Save the merged DataFrame to an Excel file
        output_file_path = os.path.join(positives_dir, "Positives1.xlsx")
        merged_df.to_excel(output_file_path, index=False, engine='openpyxl')

        print(f"Merged file saved as: {output_file_path}")

    def calculate_performance_metrics(output_dir, tumor_types, total_positive_patients, num_combinations):
        # Load data from Excel files
        positives_file_path = os.path.join(output_dir, 'Positives', 'Positives1.xlsx')
        averages_file_path = os.path.join(output_dir, 'Average1.xlsx')

        positives_df = pd.read_excel(positives_file_path)
        averages_df = pd.read_excel(averages_file_path)

        # Get the list of Positive Count columns
        positive_count_columns = [col for col in positives_df.columns if col.startswith("Positive Det")]

        # Total number of positive patients across all Cohorts
        total_patients = positives_df["Total Positive Patients"].sum()

        # Loop through each Positive Count column
        for i, column in enumerate(positive_count_columns, start=1):
            if i > num_combinations:
                break  # Stop after processing the specified number of combinations
            
            # Determine the Cohort type based on the highest value in the corresponding "Average Distance %(3)-i" column
            avg_column = f"Average Distance %(3)-{i}"
            
            if avg_column not in averages_df.columns:
                print(f"Column {avg_column} not found in Averages file, skipping...")
                continue  # Skip this iteration if column is missing
            
            max_value_row = averages_df[avg_column].idxmax()
            Cohort_type = averages_df.loc[max_value_row, "Cohort"]

            # Ensure the Cohort exists in positives_df
            Cohort_row = positives_df[positives_df["Cohort"] == Cohort_type]
            if Cohort_row.empty:
                print(f"Cohort {Cohort_type} not found in Positives file, skipping...")
                continue
            
            Cohort_row = Cohort_row.iloc[0]  # Convert to Series
            
            # Extract necessary values
            TP = Cohort_row[column]  # True Positives
            FN = Cohort_row["Total Positive Patients"] - TP  # False Negatives
            FP = positives_df[column].sum() - TP  # False Positives
            TN = total_patients - (TP + FP + FN)  # True Negatives

            # Calculate Performance Metrics
            sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
            accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            f1_score = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
            
            # Write results into the DataFrame
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Sensitivity {i}"] = sensitivity
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Accuracy {i}"] = accuracy
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Precision {i}"] = precision
            positives_df.loc[positives_df["Cohort"] == Cohort_type, f"F1 Score {i}"] = f1_score

        # Save the results back to the Excel file
        positives_df.to_excel(positives_file_path, index=False, engine='openpyxl')

        print(f"Results saved to: {positives_file_path}")

    def get_top_5_metrics(output_dir, tumor_types, num_combinations):
        excel_file_path = os.path.join(output_dir, 'Positives', 'Positives1.xlsx')
        csv_file_folder = output_dir
        output_path = os.path.join(output_dir, 'Positives')

        # Read the Excel file
        excel_df = pd.read_excel(excel_file_path)

        # Function for natural sorting (e.g., 1,2,3 instead of 1,10,100)
        def natural_sort_key(text):
            return [int(num) if num.isdigit() else num for num in re.split(r'(\d+)', text)]

        # Function to get selected biomarkers from the appropriate CSV file
        def get_selected_biomarkers(csv_file_name, col_start=1, col_end=4):
            csv_file_path = os.path.join(csv_file_folder, csv_file_name)
            if os.path.exists(csv_file_path):
                csv_df = pd.read_csv(csv_file_path)
                return csv_df.iloc[-1, col_start:col_end].values.tolist()  # Convert NumPy array to Python list
            else:
                print(f"Warning: File {csv_file_name} not found.")
                return ["NA"]  # Return "NA" instead of an empty list

        # Initialize a list to store results
        results = []

        # Process each Cohort (row) in the Excel file
        for idx, row in excel_df.iterrows():
            Cohort = row['Cohort']
            print(f"Processing Cohort: {Cohort}")
            
            # Initialize the list for this Cohort
            Cohort_results = {
                'Cohort': Cohort,
                'Top F1 Score 1': 'NA', 'Top Accuracy 1': 'NA', 'Top Precision 1': 'NA', 'Top Sensitivity 1': 'NA', 'Top Biomarkers 1': 'NA',
                'Top F1 Score 2': 'NA', 'Top Accuracy 2': 'NA', 'Top Precision 2': 'NA', 'Top Sensitivity 2': 'NA', 'Top Biomarkers 2': 'NA',
                'Top F1 Score 3': 'NA', 'Top Accuracy 3': 'NA', 'Top Precision 3': 'NA', 'Top Sensitivity 3': 'NA', 'Top Biomarkers 3': 'NA',
                'Top F1 Score 4': 'NA', 'Top Accuracy 4': 'NA', 'Top Precision 4': 'NA', 'Top Sensitivity 4': 'NA', 'Top Biomarkers 4': 'NA',
                'Top F1 Score 5': 'NA', 'Top Accuracy 5': 'NA', 'Top Precision 5': 'NA', 'Top Sensitivity 5': 'NA', 'Top Biomarkers 5': 'NA'
            }
            
            # Generate ordered metric column pairs
            metric_columns = []
            for i in range(1, num_combinations + 1):
                metric_columns.append({
                    'f1_score': f'F1 Score {i}',
                    'accuracy': f'Accuracy {i}',
                    'precision': f'Precision {i}',
                    'sensitivity': f'Sensitivity {i}'
                })
            
            # Store valid metric combinations
            valid_metrics = []
            
            for metrics in metric_columns:
                if not all(col in row for col in metrics.values()):
                    continue  # Skip missing columns
                
                f1_value = row[metrics['f1_score']]
                accuracy_value = row[metrics['accuracy']]
                precision_value = row[metrics['precision']]
                sensitivity_value = row[metrics['sensitivity']]
                
                if pd.notna(accuracy_value) and accuracy_value > 0:
                    # Determine which CSV file to read based on the metric number
                    metric_num = int(metrics['f1_score'].split(' ')[2])
                    csv_file_name = f'FeatureVectorTissueDet{metric_num}.csv'
                    
                    # Get selected biomarkers for the current metric
                    biomarkers = get_selected_biomarkers(csv_file_name)
                    
                    # Store the result
                    valid_metrics.append({
                        'f1_score': f1_value,
                        'accuracy': accuracy_value,
                        'precision': precision_value,
                        'sensitivity': sensitivity_value,
                        'Biomarkers': biomarkers
                    })
            
            # If valid metrics exist, get the top 5 sorted by highest F1 score
            if valid_metrics:
                sorted_metrics = sorted(valid_metrics, key=lambda x: x['f1_score'], reverse=True)[:5]
                
                # Update the Cohort result with the top 5 metrics
                for i, res in enumerate(sorted_metrics):
                    Cohort_results[f'Top F1 Score {i + 1}'] = res['f1_score']
                    Cohort_results[f'Top Accuracy {i + 1}'] = res['accuracy']
                    Cohort_results[f'Top Precision {i + 1}'] = res['precision']
                    Cohort_results[f'Top Sensitivity {i + 1}'] = res['sensitivity']
                    Cohort_results[f'Top Biomarkers {i + 1}'] = res['Biomarkers']
            
            # Append the result for the current Cohort
            results.append(Cohort_results)

        # Create a DataFrame for the results
        results_df = pd.DataFrame(results)

        # Ensure output path exists
        os.makedirs(output_path, exist_ok=True)

        # Define output file path
        output_file_path = os.path.join(output_path, 'Top_5_Metrics.xlsx')

        # Save the results to Excel
        results_df.to_excel(output_file_path, index=False, engine='openpyxl')

        print(f"Results saved to {output_file_path}")

    def get_specific_performance(output_dir, specific_cohort, specific_biomarkers):
        """Get performance metrics for a specific cohort and biomarker combination"""
        # Load the Top_5_Metrics.xlsx file
        top_metrics_file = os.path.join(output_dir, 'Positives', 'Top_5_Metrics.xlsx')
        
        if not os.path.exists(top_metrics_file):
            print(f"Top metrics file not found: {top_metrics_file}")
            return None
        
        # Read the Excel file
        df = pd.read_excel(top_metrics_file)
        
        # Find the row for the specific cohort
        cohort_row = df[df['Cohort'] == specific_cohort]
        
        if cohort_row.empty:
            print(f"Cohort '{specific_cohort}' not found in the metrics file")
            return None
        
        # Convert the specific biomarkers to a sorted list for comparison
        specific_biomarkers_sorted = sorted(specific_biomarkers)
        
        # Check each of the top 5 biomarker combinations for a match
        for i in range(1, 6):
            biomarkers_col = f'Top Biomarkers {i}'
            f1_col = f'Top F1 Score {i}'
            accuracy_col = f'Top Accuracy {i}'
            precision_col = f'Top Precision {i}'
            sensitivity_col = f'Top Sensitivity {i}'
            
            if biomarkers_col not in cohort_row.columns:
                continue
                
            # Get the biomarkers from the file (convert string representation to list)
            file_biomarkers_str = cohort_row[biomarkers_col].values[0]
            if isinstance(file_biomarkers_str, str) and file_biomarkers_str != 'NA':
                # Convert string representation of list to actual list
                try:
                    file_biomarkers = eval(file_biomarkers_str)
                    file_biomarkers_sorted = sorted(file_biomarkers)
                    
                    # Check if biomarkers match
                    if file_biomarkers_sorted == specific_biomarkers_sorted:
                        f1_score = cohort_row[f1_col].values[0]
                        accuracy = cohort_row[accuracy_col].values[0]
                        precision = cohort_row[precision_col].values[0]
                        sensitivity = cohort_row[sensitivity_col].values[0]
                        
                        print(f"\nPerformance metrics for {specific_cohort} with biomarkers {specific_biomarkers}:")
                        print(f"F1 Score: {f1_score:.4f}")
                        print(f"Accuracy: {accuracy:.4f}")
                        print(f"Precision: {precision:.4f}")
                        print(f"Sensitivity: {sensitivity:.4f}")
                        
                        return {
                            'Cohort': specific_cohort,
                            'Biomarkers': specific_biomarkers,
                            'F1 Score': f1_score,
                            'Accuracy': accuracy,
                            'Precision': precision,
                            'Sensitivity': sensitivity
                        }
                except:
                    continue
        
        print(f"No matching biomarker combination found for {specific_cohort}")
        return None

    # ==================================================
    # Main Execution
    # ==================================================

    # Part 1: Preliminary Analysis and Formation of Average1.xlsx
    process_clinical_data(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range, offset)

    # Part 2: Formation of Positives1.xlsx and Performance Metrics
    merge_positives_to_excel(output_dir, cohorts_to_analyze, num_combinations)
    calculate_performance_metrics(output_dir, cohorts_to_analyze, total_positive_patients, num_combinations)
    get_top_5_metrics(output_dir, cohorts_to_analyze, num_combinations)

    # Get specific performance metrics if cohort and biomarkers are provided
    if specific_cohort and specific_biomarkers:
        get_specific_performance(output_dir, specific_cohort, specific_biomarkers)

# Example usage (Provide the required user-defined values)
file_path = r'D:\Cancer Detection folder\Test 3\Clinical cancer validation data.xlsx'
sheet_name = 'Normal and Cancer'
biomarkers = ['sc-kit/SCFR', 'CA-125', 'CA 19-9', 'HE4', 'Prolactin', 'SCF', 'Total PSA', 'IGFBP3']
output_dir = r"D:\Cancer Detection folder\Test 3"

tumor_types = {
    'normal': 47,
    'bladder': 9,
    'ovarian': 11,
    'pancreatic': 17
}
total_positive_patients = {
    "Bladder": 26, "Ovarian": 32, "Pancreatic": 39
}
cohorts_to_analyze = ['Bladder', 'Ovarian', 'Pancreatic']
num_combinations = 56  
random_patient_range = 322 
offset = -1  

# Specific cohort and biomarker combination to analyze
specific_cohort = 'Bladder'  # Change this to the desired cohort
specific_biomarkers = ['sc-kit/SCFR', 'HE4', 'Total PSA']  # Change this to the desired biomarker combination

# Execute the TDA_cancerDetect function
TDAcancer_detect_2_specificCohort(file_path, sheet_name, biomarkers, output_dir, tumor_types, total_positive_patients, num_combinations, random_patient_range, cohorts_to_analyze, offset, specific_cohort, specific_biomarkers)

Processing combination 1: ['sc-kit/SCFR', 'CA-125', 'CA 19-9']


C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1036108117.py:249: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, selected_biomarkers_row])


Saved: D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet1.csv
Processing combination 2: ['sc-kit/SCFR', 'CA-125', 'HE4']


C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1036108117.py:249: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, selected_biomarkers_row])


Saved: D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet2.csv
Processing combination 3: ['sc-kit/SCFR', 'CA-125', 'Prolactin']


C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1036108117.py:249: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, selected_biomarkers_row])


... (105 outputs left out)

C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1036108117.py:249: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_combined, selected_biomarkers_row])


Saved: D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet56.csv
Processed and updated D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet1.csv and D:\Cancer Detection folder\Test 3\AverageDet1.csv successfully.
Processed and updated D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet2.csv and D:\Cancer Detection folder\Test 3\AverageDet2.csv successfully.
Processed and updated D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet3.csv and D:\Cancer Detection folder\Test 3\AverageDet3.csv successfully.
Processed and updated D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet4.csv and D:\Cancer Detection folder\Test 3\AverageDet4.csv successfully.
Processed and updated D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet5.csv and D:\Cancer Detection folder\Test 3\AverageDet5.csv successfully.
Processed and updated D:\Cancer Detection folder\Test 3\FeatureVectorTissueDet6.csv and D:\Cancer Detection folder\Test 3\AverageDet6.csv successfully.
Processed and upda

C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1036108117.py:417: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positives_df.loc[positives_df["Cohort"] == Cohort_type, f"Precision {i}"] = precision
C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1036108117.py:418: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  positives_df.loc[positives_df["Cohort"] == Cohort_type, f"F1 Score {i}"] = f1_score
C:\Users\Sudarshan Gogoi\AppData\Local\Temp\ipykernel_10196\1036108117.py:415: PerformanceWarning: DataFrame is hi

Results saved to: D:\Cancer Detection folder\Test 3\Positives\Positives1.xlsx
Processing Cohort: Bladder
Processing Cohort: Ovarian
Processing Cohort: Pancreatic
Results saved to D:\Cancer Detection folder\Test 3\Positives\Top_5_Metrics.xlsx

Performance metrics for Bladder with biomarkers ['sc-kit/SCFR', 'HE4', 'Total PSA']:
F1 Score: 0.6087
Accuracy: 0.7216
Precision: 0.4884
Sensitivity: 0.8077
